In [27]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_text_ingest_repo_root(start: Path) -> Path | None:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "data_ingestion").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    return None


REPO_ROOT = find_text_ingest_repo_root(Path.cwd())
if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the local text-ingest repo. Run this notebook from "
        "the repo root "
        "or from a subfolder such as examples/."
    )

SRC_PATH = str(REPO_ROOT / "src")
if SRC_PATH in sys.path:
    sys.path.remove(SRC_PATH)
sys.path.insert(0, SRC_PATH)

for module_name in list(sys.modules):
    if module_name == "data_ingestion" or module_name.startswith("data_ingestion."):
        del sys.modules[module_name]

import data_ingestion

print("Repo root:", REPO_ROOT)
print("Using data_ingestion from:", Path(data_ingestion.__file__).resolve())
print("Expected local src path:", SRC_PATH)

Repo root: /home/daniilmiheev/p/de/textDump/data_ingestion
Using data_ingestion from: /home/daniilmiheev/p/de/textDump/data_ingestion/src/data_ingestion/__init__.py
Expected local src path: /home/daniilmiheev/p/de/textDump/data_ingestion/src


In [ ]:
import collections.abc
import contextlib
import hashlib
import json
import math
import re
import time
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import numpy as np
import polars as pl

TRUE_STRINGS = {"1", "true", "yes", "y", "on"}
FALSE_STRINGS = {"0", "false", "no", "n", "off"}


def env_str(name: str, default: str) -> str:
    return os.getenv(name, default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    value = raw.strip().lower()
    if value in TRUE_STRINGS:
        return True
    if value in FALSE_STRINGS:
        return False
    valid_values = sorted(TRUE_STRINGS | FALSE_STRINGS)
    raise ValueError(
        f"Invalid boolean env var {name}={raw!r}; expected one of {valid_values}"
    )


def env_int(name: str, default: int) -> int:
    return int(os.getenv(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.getenv(name, str(default)))


def parse_date(value: Any, default: date | None = None) -> date:
    """Parse common date/datetime strings with stdlib datetime."""
    if isinstance(value, date) and not isinstance(value, datetime):
        return value
    if isinstance(value, datetime):
        return value.date()
    if value is None:
        return default or date.today()
    text = str(value).strip()
    if not text or text.lower() in {"none", "nan", "nat"}:
        return default or date.today()
    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00")).date()
    except Exception:
        pass
    try:
        return date.fromisoformat(text[:10])
    except Exception:
        return default or date.today()


def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return isinstance(value, str) and value.strip().lower() in {
        "",
        "none",
        "nan",
        "nat",
    }


def to_float(value: Any, default: float = 0.0) -> float:
    if is_missing(value):
        return default
    try:
        out = float(value)
        return out if math.isfinite(out) else default
    except Exception:
        return default


def safe_ratio(num: Any, den: Any) -> float:
    den_f = to_float(den, np.nan)
    if not math.isfinite(den_f) or den_f == 0:
        return np.nan
    return to_float(num, np.nan) / den_f


def pl_empty() -> pl.DataFrame:
    return pl.DataFrame([])


def ensure_columns(
    df: pl.DataFrame, columns: collections.abc.Iterable[str], default: Any = None
) -> pl.DataFrame:
    if df.is_empty() and not df.columns:
        df = pl.DataFrame({col: [] for col in columns})
    missing = [col for col in columns if col not in df.columns]
    if missing:
        df = df.with_columns([pl.lit(default).alias(col) for col in missing])
    return df


def safe_read_csv_polars(path: Path) -> pl.DataFrame:
    if not path.exists():
        return pl_empty()
    try:
        return pl.read_csv(path, infer_schema_length=10000, ignore_errors=True)
    except Exception:
        return pl_empty()


def concat_polars(frames: list[pl.DataFrame]) -> pl.DataFrame:
    frames = [
        f
        for f in frames
        if isinstance(f, pl.DataFrame) and (not f.is_empty() or f.columns)
    ]
    if not frames:
        return pl_empty()
    return pl.concat(frames, how="diagonal_relaxed")


def rows_to_polars(
    rows: list[dict[str, Any]], columns: collections.abc.Iterable[str] | None = None
) -> pl.DataFrame:
    if rows:
        return pl.from_dicts(rows, infer_schema_length=None)
    if columns:
        return pl.DataFrame({c: [] for c in columns})
    return pl_empty()


def csv_safe_value(value: Any, list_separator: str | None = None) -> Any:
    if isinstance(value, list) and list_separator is not None:
        return list_separator.join(str(x) for x in value)
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


def dataframe_for_csv(
    df: pl.DataFrame, list_separator: str | None = None
) -> pl.DataFrame:
    if df.is_empty():
        return pl.DataFrame({col: [] for col in df.columns})
    rows = [
        {
            col: csv_safe_value(value, list_separator=list_separator)
            for col, value in row.items()
        }
        for row in df.to_dicts()
    ]
    return rows_to_polars(rows, columns=df.columns)


def display_head(df: pl.DataFrame, n: int = 20) -> None:
    display(df.head(n))


def slugify(value: object, max_len: int | None = None) -> str:
    """Convert arbitrary text into a safe short filename slug."""
    max_len = SLUG_MAX_LEN if max_len is None else max_len
    text = str(value or UNKNOWN_VALUE).lower().strip()
    text = re.sub(SLUG_ALLOWED_CHARS_REGEX, SLUG_SEPARATOR, text).strip(SLUG_SEPARATOR)
    text = re.sub(SLUG_REPEAT_SEPARATOR_REGEX, SLUG_SEPARATOR, text)
    return text[:max_len].strip(SLUG_SEPARATOR) or UNKNOWN_VALUE


RUN_DATE = env_str("RUN_DATE", date.today().isoformat())
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = env_str("RUN_ID", f"{RUN_DATE}_{RUN_TS}")

INDUSTRY = env_str("INDUSTRY", "marketing agency")
INDUSTRY_DESCRIPTION = env_str(
    "INDUSTRY_DESCRIPTION",
    "Marketing, advertising, media, creative, performance-marketing, and "
    "martech agencies affected by AI adoption, client budget pressure, "
    "privacy regulation, and platform changes.",
)

RUN_MODE = env_str("RUN_MODE", "discovery").lower().strip()
BACKFILL_MODE = env_bool("BACKFILL_MODE", False)

END_DATE = env_str("END_DATE", RUN_DATE)
BACKFILL_LOOKBACK_DAYS = env_int("BACKFILL_LOOKBACK_DAYS", 3)
DISCOVERY_LOOKBACK_DAYS = env_int("DISCOVERY_LOOKBACK_DAYS", 1)
MEASUREMENT_LOOKBACK_DAYS = env_int("MEASUREMENT_LOOKBACK_DAYS", 2)

if BACKFILL_MODE:
    START_DATE = env_str(
        "START_DATE",
        (parse_date(END_DATE) - timedelta(days=BACKFILL_LOOKBACK_DAYS)).isoformat(),
    )
else:
    DEFAULT_LOOKBACK_DAYS = (
        DISCOVERY_LOOKBACK_DAYS
        if RUN_MODE == "discovery"
        else MEASUREMENT_LOOKBACK_DAYS
    )
    START_DATE = env_str(
        "START_DATE",
        (parse_date(END_DATE) - timedelta(days=DEFAULT_LOOKBACK_DAYS)).isoformat(),
    )

DEFAULT_MAX_PAGES_BY_MODE = {"backfill": 25, "discovery": 10, "measurement": 5}
DEFAULT_QUERY_VARIANTS_BY_MODE = {"discovery": 4, "measurement": 2}
DEFAULT_MAX_PAGES = (
    DEFAULT_MAX_PAGES_BY_MODE["backfill"]
    if BACKFILL_MODE
    else DEFAULT_MAX_PAGES_BY_MODE.get(RUN_MODE, 5)
)
MAX_PAGES = env_int("MAX_PAGES", DEFAULT_MAX_PAGES)
MAX_QUERY_VARIANTS = env_int(
    "MAX_QUERY_VARIANTS", DEFAULT_QUERY_VARIANTS_BY_MODE.get(RUN_MODE, 2)
)


ENABLE_AHO_RECORD_AUDIT = env_bool("ENABLE_AHO_RECORD_AUDIT", True)
INCLUDE_WEBSITE_SOURCES = env_bool("INCLUDE_WEBSITE_SOURCES", False)


ENABLE_PROGRESS_LOGGING = env_bool("ENABLE_PROGRESS_LOGGING", True)
INGESTION_LOG_EVERY_SPECS = env_int("INGESTION_LOG_EVERY_SPECS", 1)


ENABLE_PARALLEL_INGESTION = env_bool("ENABLE_PARALLEL_INGESTION", True)
MAX_INGESTION_WORKERS = max(1, env_int("MAX_INGESTION_WORKERS", 2))


ENABLE_HARD_FETCH_TIMEOUT = env_bool("ENABLE_HARD_FETCH_TIMEOUT", True)
INGESTION_SPEC_TIMEOUT_SECONDS = max(15, env_int("INGESTION_SPEC_TIMEOUT_SECONDS", 180))
INGESTION_POLL_SECONDS = max(1, env_int("INGESTION_POLL_SECONDS", 2))
INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS = max(
    1, env_int("INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS", 15)
)
INGESTION_MULTIPROCESS_START_METHOD = env_str(
    "INGESTION_MULTIPROCESS_START_METHOD", "fork"
)


DEDUPLICATE_IDENTICAL_FETCH_SPECS = env_bool("DEDUPLICATE_IDENTICAL_FETCH_SPECS", True)


SERIAL_INGESTION_SOURCES = {
    s.strip()
    for s in env_str("SERIAL_INGESTION_SOURCES", "edgar,federalregister").split(",")
    if s.strip()
}


REMOVED_SOURCES = {"github", "stackexchange"}
RATE_LIMIT_SENSITIVE_SOURCES: set[str] = set()
SOURCE_ENABLED_OVERRIDES: dict[str, bool] = {}
SOURCE_SPEC_LIMITS: dict[str, int] = {}
SOURCE_MAX_PAGES_OVERRIDES: dict[str, int] = {}
SOURCE_PAGE_SIZE_OVERRIDES: dict[str, int] = {}

PROGRESS_ESTIMATE_MIN_COMPLETED = env_int("PROGRESS_ESTIMATE_MIN_COMPLETED", 2)
PROGRESS_MESSAGE_SEPARATOR = " | "
PROGRESS_UNKNOWN_REMAINING = "unknown"


OUT_DIR = Path(env_str("OUT_DIR", "data/industry_signal_intelligence"))
RUNS_SUBDIR = "runs"
RUN_DATE_PARTITION_PREFIX = "dt="
RUN_ID_PARTITION_PREFIX = "run_id="
RAW_SPEC_SUBDIR = "raw_by_spec"
CLEAN_SUBDIR = "clean"
REPORT_SUBDIR = "reports"
CACHE_SUBDIR = "cache"

OUTPUT_FILENAMES = {
    "combined_raw_jsonl": "raw_combined_with_spec_metadata.jsonl",
    "fetcher_specs_csv": "fetcher_specs.csv",
    "coverage_csv": "fetcher_coverage.csv",
    "clean_records_csv": "clean_records.csv",
    "panel_csv": "daily_signal_panel_current_run.csv",
    "panel_history_csv": "panel_history.csv",
    "statistical_signals_csv": "statistical_signal_candidates.csv",
    "watchlist_csv": "analyst_watchlist.csv",
    "human_review_csv": "human_review_queue.csv",
    "external_outcomes_csv": "external_outcomes_template.csv",
    "report_md": "industry_signal_report.md",
}

RUN_DIR = (
    OUT_DIR
    / RUNS_SUBDIR
    / f"{RUN_DATE_PARTITION_PREFIX}{RUN_DATE}"
    / f"{RUN_ID_PARTITION_PREFIX}{RUN_ID}"
)
RAW_SPEC_DIR = RUN_DIR / RAW_SPEC_SUBDIR
CLEAN_DIR = RUN_DIR / CLEAN_SUBDIR
REPORT_DIR = RUN_DIR / REPORT_SUBDIR
CACHE_DIR = OUT_DIR / CACHE_SUBDIR
for p in [RUN_DIR, RAW_SPEC_DIR, CLEAN_DIR, REPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

COMBINED_RAW_JSONL = RUN_DIR / OUTPUT_FILENAMES["combined_raw_jsonl"]
FETCHER_SPECS_CSV = RUN_DIR / OUTPUT_FILENAMES["fetcher_specs_csv"]
COVERAGE_CSV = RUN_DIR / OUTPUT_FILENAMES["coverage_csv"]
CLEAN_RECORDS_CSV = CLEAN_DIR / OUTPUT_FILENAMES["clean_records_csv"]
PANEL_CSV = CLEAN_DIR / OUTPUT_FILENAMES["panel_csv"]
PANEL_HISTORY_CSV = OUT_DIR / OUTPUT_FILENAMES["panel_history_csv"]
SIGNIFICANT_SIGNALS_CSV = CLEAN_DIR / OUTPUT_FILENAMES["statistical_signals_csv"]
WATCHLIST_CSV = CLEAN_DIR / OUTPUT_FILENAMES["watchlist_csv"]
HUMAN_REVIEW_CSV = OUT_DIR / OUTPUT_FILENAMES["human_review_csv"]
EXTERNAL_OUTCOMES_CSV = OUT_DIR / OUTPUT_FILENAMES["external_outcomes_csv"]
REPORT_MD = REPORT_DIR / OUTPUT_FILENAMES["report_md"]


ENABLE_FETCH_CACHE = env_bool("ENABLE_FETCH_CACHE", True)
FORCE_REFETCH = env_bool("FORCE_REFETCH", False)
REUSE_EMPTY_FETCH_CACHE = env_bool("REUSE_EMPTY_FETCH_CACHE", False)
FETCH_CACHE_SUBDIR = env_str("FETCH_CACHE_SUBDIR", "fetch_cache")
FETCH_CACHE_DIR = CACHE_DIR / FETCH_CACHE_SUBDIR
FETCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FETCH_CACHE_MANIFEST_CSV = CACHE_DIR / env_str(
    "FETCH_CACHE_MANIFEST_FILENAME", "fetch_cache_manifest.csv"
)
FETCH_CACHE_SCHEMA_VERSION = env_str("FETCH_CACHE_SCHEMA_VERSION", "v1")
FETCH_CACHE_HASH_LENGTH = env_int("FETCH_CACHE_HASH_LENGTH", 32)
FETCH_CACHE_SECRET_KEYS = {"api_key", "token", "access_token", "secret", "password"}
FETCH_CACHE_FILENAME_TEMPLATE = "{source}_{cache_key}.jsonl"
FETCH_CACHE_STATUS_HIT = "cache_hit"
FETCH_CACHE_STATUS_MISS = "fetched"
FETCH_CACHE_STATUS_FAILED = "failed"
FETCH_CACHE_STATUS_TIMEOUT = "timeout"


def sanitize_for_cache(value: Any) -> Any:
    """Return a deterministic, non-secret representation of nested config values."""
    if isinstance(value, dict):
        clean = {}
        for k, v in value.items():
            key = str(k)
            if key.lower() in FETCH_CACHE_SECRET_KEYS:
                clean[key] = "<set>" if v else "<missing>"
            else:
                clean[key] = sanitize_for_cache(v)
        return clean
    if isinstance(value, (list, tuple)):
        return [sanitize_for_cache(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    return value


def stable_json_hash(payload: Any, length: int | None = None) -> str:
    length = FETCH_CACHE_HASH_LENGTH if length is None else length
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:length]


def make_fetch_cache_payload(source: str, config: dict[str, Any]) -> dict[str, Any]:
    return {
        "schema_version": FETCH_CACHE_SCHEMA_VERSION,
        "source": source,
        "config": sanitize_for_cache(config),
    }


def make_fetch_cache_key(source: str, config: dict[str, Any]) -> str:
    return stable_json_hash(make_fetch_cache_payload(source, config))


def fetch_cache_path(source: str, cache_key: str) -> Path:
    return FETCH_CACHE_DIR / FETCH_CACHE_FILENAME_TEMPLATE.format(
        source=slugify(source, 40), cache_key=cache_key
    )


def format_duration(seconds: float | int | None) -> str:
    if seconds is None:
        return PROGRESS_UNKNOWN_REMAINING
    try:
        seconds = float(seconds)
    except Exception:
        return PROGRESS_UNKNOWN_REMAINING
    if not math.isfinite(seconds) or seconds < 0:
        return PROGRESS_UNKNOWN_REMAINING
    seconds = round(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def estimated_remaining_seconds(
    started_at: float, completed: int, total: int
) -> float | None:
    if completed < PROGRESS_ESTIMATE_MIN_COMPLETED or total <= 0 or completed <= 0:
        return None
    elapsed = time.time() - started_at
    rate = elapsed / completed
    return max(total - completed, 0) * rate


def progress_log(stage: str, completed: int, total: int, **fields: Any) -> None:
    if not ENABLE_PROGRESS_LOGGING:
        return
    pct = 0.0 if total <= 0 else min(max(completed / total * 100, 0), 100)
    parts = [f"[{stage}] {completed:,}/{total:,} ({pct:.1f}%)"]
    for k, v in fields.items():
        parts.append(f"{k}={v}")
    print(PROGRESS_MESSAGE_SEPARATOR.join(parts))


EXCLUDED_SOURCES = {"wikipedia", "github", "stackexchange"}
NON_API_WEBSITE_SOURCES = {"google_news"}

MEASUREMENT_SOURCES = [
    "newsapi",
    "guardian",
    "edgar",
    "federalregister",
    "reddit",
    "hackernews",
]
DISCOVERY_EXTRA_SOURCES = ["openalex", "crossref", "openlibrary"]

SOURCE_FAMILY = {
    "newsapi": "news",
    "guardian": "news",
    "google_news": "news",
    "hackernews": "community",
    "reddit": "community",
    "federalregister": "regulatory",
    "edgar": "company_filings",
    "openalex": "research",
    "crossref": "research",
    "openlibrary": "reference",
}
DEFAULT_SOURCE_WEIGHT = 0.50
SOURCE_WEIGHT = {
    "federalregister": 1.00,
    "edgar": 0.95,
    "newsapi": 0.90,
    "guardian": 0.85,
    "google_news": 0.80,
    "hackernews": 0.60,
    "reddit": 0.55,
    "openalex": 0.45,
    "crossref": 0.40,
    "openlibrary": 0.20,
}
API_KEY_ENV_BY_SOURCE = {"newsapi": "NEWSAPI_KEY", "guardian": "GUARDIAN_API_KEY"}
HARD_SKIP_WHEN_MISSING_API_KEY = {"newsapi", "guardian"}

BASE_FETCHER_CONFIG = {}
SOURCE_FETCHER_KWARGS = {
    "newsapi": {"page_size": 100, "language": "en", "sort_by": "publishedAt"},
    "guardian": {"page_size": 50, "show_fields": "trailText,bodyText"},
    "edgar": {"limit": 100},
    "federalregister": {"per_page": 100},
    "reddit": {"limit": 100, "sort": "new"},
    "hackernews": {"hits_per_page": 100},
    "openalex": {"per_page": 100},
    "crossref": {"rows": 100},
    "openlibrary": {"limit": 100},
    "google_news": {"limit": 100},
}
QUERY_VARIANT_LIMIT_BY_MODE = {
    "discovery": MAX_QUERY_VARIANTS,
    "measurement": MAX_QUERY_VARIANTS,
}


MARKETING_INDUSTRY_DETECT_TERMS = [
    "marketing",
    "advertising",
    "media agency",
    "creative agency",
    "agency",
]
MARKETING_QUERY_ANCHOR = (
    "marketing agency OR advertising agency OR media agency OR creative agency"
)
MARKETING_SIGNAL_PLAN = {
    "industry_terms": [
        "marketing agency",
        "advertising agency",
        "media agency",
        "creative agency",
        "performance marketing agency",
        "digital agency",
        "ad agency",
        "agency holding company",
        "adtech",
        "martech",
        "programmatic advertising",
        "media buying",
        "brand marketing",
        "campaign measurement",
        "marketing attribution",
        "CMO budget",
        "marketing spend",
    ],
    "adjacent_terms": [
        "Google Ads",
        "Meta ads",
        "TikTok ads",
        "Amazon Ads",
        "cookie deprecation",
        "privacy sandbox",
        "generative AI",
        "creative automation",
    ],
}
GENERIC_SIGNAL_PLAN_TEMPLATE = {
    "industry_terms": [
        "{industry}",
        "{industry} industry",
        "{industry} companies",
        "{industry} market",
    ],
    "adjacent_terms": [
        "regulation",
        "AI",
        "automation",
        "funding",
        "layoffs",
        "customer demand",
        "supply chain",
    ],
}
COMMON_CHALLENGE_TERMS = [
    "headwind",
    "challenge",
    "barrier",
    "constraint",
    "bottleneck",
    "margin pressure",
    "pricing pressure",
    "budget cuts",
    "layoffs",
    "lawsuit",
    "enforcement",
    "regulatory burden",
    "compliance cost",
    "client churn",
    "demand slowdown",
    "platform change",
    "privacy",
    "fraud",
    "measurement",
    "uncertainty",
]
COMMON_OPPORTUNITY_TERMS = [
    "growth",
    "partnership",
    "funding",
    "acquisition",
    "launch",
    "new product",
    "adoption",
    "approval",
    "patent",
    "open source",
    "contract",
    "expansion",
    "demand",
    "investment",
    "efficiency",
    "automation",
]
COMMON_RISK_TERMS = [
    "ban",
    "fine",
    "lawsuit",
    "recall",
    "shortage",
    "disruption",
    "tariff",
    "investigation",
    "probe",
    "decline",
    "cuts",
    "layoff",
    "breach",
    "fraud",
    "complaint",
    "enforcement",
    "risk factor",
]
STRONG_RISK_TERMS = [
    "ban",
    "fine",
    "lawsuit",
    "recall",
    "shortage",
    "disruption",
    "tariff",
    "investigation",
    "probe",
    "decline",
    "layoff",
    "layoffs",
    "breach",
    "fraud",
    "complaint",
    "enforcement",
    "risk factor",
    "budget cut",
    "budget cuts",
    "margin pressure",
    "client churn",
    "demand slowdown",
]
STRONG_OPPORTUNITY_TERMS = [
    "growth",
    "partnership",
    "funding",
    "acquisition",
    "launch",
    "new product",
    "adoption",
    "approval",
    "contract",
    "expansion",
    "investment",
    "efficiency",
    "automation",
    "customer growth",
    "tailwind",
]
AMBIGUOUS_RISK_TERMS = [
    "regulation",
    "regulatory",
    "pressure",
    "challenge",
    "measurement",
    "privacy",
    "uncertainty",
    "cost",
]
AMBIGUOUS_OPPORTUNITY_TERMS = [
    "demand",
    "open source",
    "patent",
    "market",
    "approval",
    "automation",
]
DIRECTION_NEGATION_TERMS = [
    "no",
    "not",
    "without",
    "avoid",
    "avoids",
    "avoided",
    "reduce",
    "reduces",
    "reduced",
    "lower",
    "lowers",
    "lowered",
    "mitigate",
    "mitigates",
    "mitigated",
    "less",
    "fewer",
]
DIRECTION_RESOLUTION_MARGIN = env_float("DIRECTION_RESOLUTION_MARGIN", 0.75)
DIRECTION_MIN_EVIDENCE_SCORE = env_float("DIRECTION_MIN_EVIDENCE_SCORE", 1.0)
DIRECTION_CONTEXT_CHARS = env_int("DIRECTION_CONTEXT_CHARS", 28)

DEFAULT_QUERY_LENSES = [
    {"name": "core_industry", "query_template": "{industry}", "priority": 1},
    {
        "name": "ai_automation",
        "query_template": "{industry} AI automation workflow tool",
        "priority": 1,
    },
    {
        "name": "budget_demand",
        "query_template": "{industry} budget spend demand clients",
        "priority": 1,
    },
    {
        "name": "policy_regulation",
        "query_template": "{industry} regulation privacy compliance enforcement",
        "priority": 1,
    },
    {
        "name": "platform_changes",
        "query_template": "{industry} Google Meta TikTok Amazon platform changes",
        "priority": 2,
    },
    {
        "name": "competition_ma",
        "query_template": (
            "{industry} acquisition merger partnership investment competition"
        ),
        "priority": 2,
    },
    {
        "name": "labor_margin",
        "query_template": "{industry} layoffs hiring margin pressure cost reduction",
        "priority": 2,
    },
    {
        "name": "technical_adoption",
        "query_template": "{industry} open source API tool workflow developer",
        "priority": 3,
    },
]
DEFAULT_SIGNAL_TAXONOMY = {
    "policy_regulation": [
        "regulation",
        "regulatory",
        "rule",
        "law",
        "compliance",
        "guidance",
        "enforcement",
        "fine",
        "privacy",
        "cookie",
        "antitrust",
    ],
    "ai_technology": [
        "AI",
        "generative AI",
        "automation",
        "model",
        "agent",
        "workflow",
        "creative automation",
        "measurement",
        "attribution",
        "open source",
    ],
    "market_competition": [
        "market",
        "competition",
        "competitor",
        "pricing",
        "merger",
        "acquisition",
        "partnership",
        "investment",
        "funding",
        "revenue",
    ],
    "demand_budget": [
        "budget",
        "spend",
        "CMO",
        "procurement",
        "demand",
        "customer",
        "client",
        "churn",
        "renewal",
        "RFP",
    ],
    "labor_operations": [
        "layoff",
        "hiring",
        "talent",
        "margin",
        "cost",
        "utilization",
        "productivity",
        "offshore",
        "delivery",
    ],
    "legal_reputation": [
        "lawsuit",
        "fraud",
        "brand safety",
        "misinformation",
        "copyright",
        "FTC",
        "complaint",
        "investigation",
    ],
}
CATEGORY_PRIORITY = [
    "policy_regulation",
    "legal_reputation",
    "ai_technology",
    "demand_budget",
    "market_competition",
    "labor_operations",
]
UNCATEGORIZED_CATEGORY = "uncategorized"
GENERAL_CATEGORY = "general_industry_update"


UNKNOWN_VALUE = "unknown"
JSON_ENCODING = "utf-8"
SPACE_REGEX = r"\s+"
URL_REGEX = r"https?://\S+"
NON_ALNUM_REGEX = r"[^a-z0-9]+"
WWW_PREFIX_REGEX = r"^www\."
TERM_LEFT_BOUNDARY = r"(?<![a-z0-9])"
TERM_RIGHT_BOUNDARY = r"(?![a-z0-9])"
HASH_JOINER = "||"
HASH_LENGTH = 24
SLUG_MAX_LEN = 80
SLUG_ALLOWED_CHARS_REGEX = r"[^a-z0-9]+"
SLUG_REPEAT_SEPARATOR_REGEX = r"_+"
SLUG_SEPARATOR = "_"
SPEC_FILENAME_TEMPLATE = "spec_{spec_idx:04d}_{source}_{lens_slug}.jsonl"

RAW_REQUIRED_COLUMNS = [
    "source",
    "external_id",
    "title",
    "abstract",
    "full_text",
    "url",
    "topic",
    "record_type",
    "published_date",
    "fetched_at",
]
TEXT_COLUMNS_FOR_AUDIT = ["title", "abstract", "source", "domain"]
INCLUDE_FULL_TEXT_IN_AUDIT = env_bool("INCLUDE_FULL_TEXT_IN_AUDIT", False)
AUDIT_TEXT_MAX_CHARS = env_int("AUDIT_TEXT_MAX_CHARS", 600)
FULL_TEXT_AUDIT_CHARS = AUDIT_TEXT_MAX_CHARS
FULL_TEXT_DEDUPE_CAP = 5000
FULL_TEXT_DEDUPE_DIVISOR = 50000
DEDUPE_URL_BONUS = 0.10
DEFAULT_DEDUPE_SOURCE_WEIGHT = 0.50
DEFAULT_MISSING_AGE_DAYS = 365
FRESHNESS_HALFLIFE_DAYS = 90.0

TRIAGE_SCORE_SCALE = 100
RULE_RELEVANCE_INDUSTRY_WEIGHT = 2
RULE_TRIAGE_WEIGHTS = {
    "relevance": 0.30,
    "freshness": 0.25,
    "signal_strength": 0.25,
    "source_weight": 0.20,
}
LOW_VOLUME_WARNING_THRESHOLD = 500
PIPELINE_RUNTIME_OPTIONS = {
    "enrich_full_text": False,
    "write_raw_payload": False,
    "drop_raw_payload_after_transform": True,
    "sink_write_batch_size": 500,
}


AHO_MIN_TEXT_CHARS = env_int("AHO_MIN_TEXT_CHARS", 20)
AHO_MIN_PATTERN_CHARS = env_int("AHO_MIN_PATTERN_CHARS", 3)
AHO_MAX_PATTERNS_PER_GROUP = env_int("AHO_MAX_PATTERNS_PER_GROUP", 120)
AHO_CATEGORY_SCORE_NORMALIZER = env_float("AHO_CATEGORY_SCORE_NORMALIZER", 3.0)
AHO_DIRECTION_SCORE_NORMALIZER = env_float("AHO_DIRECTION_SCORE_NORMALIZER", 2.0)
AHO_MIN_TOTAL_MATCH_GROUPS = env_int("AHO_MIN_TOTAL_MATCH_GROUPS", 1)
AHO_TRIAGE_WEIGHTS = {"aho_confidence": 0.45, "freshness": 0.30, "source_weight": 0.25}
AHO_CONFIDENCE_WEIGHTS = {"relevance": 0.35, "category": 0.45, "direction": 0.20}
DIRECTION_PATTERNS = {
    "risk": [
        "risk",
        "threat",
        "decline",
        "disruption",
        "regulation",
        "regulatory",
        "pressure",
        "enforcement",
        "lawsuit",
        "lawsuits",
        "layoff",
        "layoffs",
        "loss",
        "budget cut",
        "budget cuts",
        "cut spending",
        "reduced spending",
        "margin compression",
        "churn",
        "slowdown",
        "constraint",
        "bottleneck",
        "challenge",
        "headwind",
    ],
    "opportunity": [
        "opportunity",
        "growth",
        "demand",
        "investment",
        "adoption",
        "expansion",
        "launch",
        "partnership",
        "acquisition",
        "funding",
        "new product",
        "efficiency",
        "automation",
        "contract",
        "customer growth",
        "market opening",
        "tailwind",
    ],
    "neutral": [
        "general update",
        "background",
        "informational",
        "announcement",
        "overview",
        "routine news",
    ],
}
AHO_DISPLAY_ROWS = 20
FINAL_CLASSIFICATION_DISPLAY_ROWS = 25
CLASSIFICATION_METHOD_LABELS = {
    "rules_and_aho": "rules_and_aho",
    "rules_only": "rules_only",
    "aho_only": "aho_only",
    "not_relevant": "not_relevant",
}


MIN_RECORDS_FOR_STATS = env_int("MIN_RECORDS_FOR_STATS", 30)
MIN_BASELINE_DAYS = env_int("MIN_BASELINE_DAYS", 14)
BASELINE_LOOKBACK_DAYS = env_int("BASELINE_LOOKBACK_DAYS", 90)
FDR_ALPHA = env_float("FDR_ALPHA", 0.10)
WILSON_Z = env_float("WILSON_Z", 1.96)
MIN_CORROBORATING_SOURCE_FAMILIES = env_int("MIN_CORROBORATING_SOURCE_FAMILIES", 2)
MIN_CORROBORATING_RECORDS = env_int("MIN_CORROBORATING_RECORDS", 10)

CLAIM_LEVELS = {
    "stat_and_corroborated": "statistically_unusual_and_corroborated",
    "stat_single_family": "statistically_unusual_single_family",
    "corroborated_directional": "corroborated_directional_signal",
    "directional_watchlist": "directional_watchlist",
    "low_volume": "exploratory_low_volume",
}


DEFAULT_DISPLAY_ROWS = 20
STAT_CANDIDATE_DISPLAY_ROWS = 25
CORROBORATION_DISPLAY_ROWS = 30
CLEAN_RECORDS_DISPLAY_ROWS = 10
WATCHLIST_DISPLAY_ROWS = 25
REVIEW_TOP_N = env_int("REVIEW_TOP_N", 100)
REVIEW_RANDOM_N = env_int("REVIEW_RANDOM_N", 100)
REVIEW_RANDOM_SEED = env_int("REVIEW_RANDOM_SEED", 42)
LABEL_TRUE_VALUES = {"true", "1", "yes"}
LABEL_FALSE_VALUES = {"false", "0", "no"}
LABEL_VALID_VALUES = LABEL_TRUE_VALUES | LABEL_FALSE_VALUES

REVIEW_COLUMNS = [
    "record_id",
    "record_fingerprint",
    "_run_date",
    "_industry",
    "source",
    "_source_family",
    "_query_lens",
    "event_date",
    "primary_category",
    "direction",
    "triage_score",
    "title",
    "abstract",
    "url",
    "matched_industry_terms",
    "matched_challenge_terms",
    "matched_risk_terms",
    "matched_opportunity_terms",
]
HUMAN_LABEL_COLUMNS = [
    "human_relevant",
    "human_category",
    "human_direction",
    "human_true_signal",
    "human_notes",
]
LIST_EXPORT_COLUMNS = [
    "matched_industry_terms",
    "matched_adjacent_terms",
    "matched_challenge_terms",
    "matched_risk_terms",
    "matched_opportunity_terms",
]
WATCHLIST_EXPORT_COLUMNS = [
    "record_id",
    "source",
    "_source_family",
    "_query_lens",
    "event_date",
    "primary_category",
    "direction",
    "triage_score",
    "relevance_score",
    "freshness_score",
    "signal_strength_score",
    "title",
    "abstract",
    "url",
    "matched_industry_terms",
    "matched_adjacent_terms",
    "matched_challenge_terms",
    "matched_risk_terms",
    "matched_opportunity_terms",
]

EXTERNAL_OUTCOME_TEMPLATE_ROWS = [
    {
        "outcome_date": "YYYY-MM-DD",
        "industry": INDUSTRY,
        "outcome_type": (
            "layoff | earnings_call | regulation | m&a | product_launch | "
            "budget_cut | other"
        ),
        "outcome_name": "",
        "outcome_direction": "risk | opportunity | neutral",
        "source_url": "",
        "notes": "",
    }
]
DATE_YYYY_MM_DD_REGEX = r"\d{4}-\d{2}-\d{2}"

REPORT_TOP_N_STATS = 20
REPORT_TOP_N_WATCHLIST = 25
REPORT_CATEGORY_SECTIONS = [
    {
        "name": "Policy/regulation records",
        "column": "primary_category",
        "value": "policy_regulation",
        "n": 15,
    },
    {
        "name": "AI/technology records",
        "column": "primary_category",
        "value": "ai_technology",
        "n": 15,
    },
    {
        "name": "Demand/budget records",
        "column": "primary_category",
        "value": "demand_budget",
        "n": 15,
    },
    {"name": "Risk records", "column": "direction", "value": "risk", "n": 15},
]
REPORT_REALITY_CHECK_TEXT = (
    "This report separates triage from evidence. Triage scores rank records "
    "for analyst review. Statistical candidates require historical baseline, "
    "adequate denominators, FDR correction, and corroboration checks."
)

PLOT_FIGSIZE_SOURCE = (9, 4)
PLOT_FIGSIZE_CATEGORY = (9, 4)
PLOT_FIGSIZE_SIGNAL_RATE = (10, 4)
PLOT_BAR_WIDTH = 0.35

print(
    json.dumps(
        {
            "industry": INDUSTRY,
            "run_mode": RUN_MODE,
            "backfill_mode": BACKFILL_MODE,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "max_pages": MAX_PAGES,
            "max_query_variants": MAX_QUERY_VARIANTS,
            "run_dir": str(RUN_DIR),
            "dataframe_engine": "polars",
            "enable_fetch_cache": ENABLE_FETCH_CACHE,
            "force_refetch": FORCE_REFETCH,
            "fetch_cache_manifest_csv": str(FETCH_CACHE_MANIFEST_CSV),
            "enable_progress_logging": ENABLE_PROGRESS_LOGGING,
            "enable_aho_record_audit": ENABLE_AHO_RECORD_AUDIT,
            "aho_min_total_match_groups": AHO_MIN_TOTAL_MATCH_GROUPS,
            "aho_max_patterns_per_group": AHO_MAX_PATTERNS_PER_GROUP,
            "aho_min_text_chars": AHO_MIN_TEXT_CHARS,
            "include_full_text_in_audit": INCLUDE_FULL_TEXT_IN_AUDIT,
            "audit_text_max_chars": AUDIT_TEXT_MAX_CHARS,
            "excluded_sources": sorted(EXCLUDED_SOURCES),
        },
        indent=2,
    )
)

{
  "industry": "marketing agency",
  "run_mode": "discovery",
  "backfill_mode": false,
  "start_date": "2025-11-03",
  "end_date": "2026-05-02",
  "max_pages": 10,
  "max_query_variants": 4,
  "run_dir": "data/industry_signal_intelligence/runs/dt=2026-05-02/run_id=2026-05-02_20260503T012200Z",
  "dataframe_engine": "polars",
  "enable_fetch_cache": true,
  "force_refetch": false,
  "fetch_cache_manifest_csv": "data/industry_signal_intelligence/cache/fetch_cache_manifest.csv",
  "enable_progress_logging": true,
  "enable_aho_record_audit": true,
  "aho_min_total_match_groups": 1,
  "aho_max_patterns_per_group": 120,
  "aho_min_text_chars": 20,
  "include_full_text_in_audit": false,
  "audit_text_max_chars": 600,
  "excluded_sources": [
    "github",
    "stackexchange",
    "wikipedia"
  ]
}


In [29]:
def fallback_signal_plan(industry: str) -> dict[str, Any]:
    """Build a deterministic signal plan from the top-level config."""
    industry_l = industry.lower()
    is_marketing_like = any(
        term in industry_l for term in MARKETING_INDUSTRY_DETECT_TERMS
    )

    if is_marketing_like:
        industry_terms = MARKETING_SIGNAL_PLAN["industry_terms"]
        adjacent_terms = MARKETING_SIGNAL_PLAN["adjacent_terms"]
    else:
        industry_terms = [
            t.format(industry=industry)
            for t in GENERIC_SIGNAL_PLAN_TEMPLATE["industry_terms"]
        ]
        adjacent_terms = [
            t.format(industry=industry)
            for t in GENERIC_SIGNAL_PLAN_TEMPLATE["adjacent_terms"]
        ]

    query_lenses = [
        {
            "name": lens["name"],
            "query": lens["query_template"].format(industry=industry),
            "priority": lens["priority"],
        }
        for lens in DEFAULT_QUERY_LENSES
    ]

    return {
        "industry": industry,
        "industry_terms": list(industry_terms),
        "adjacent_terms": list(adjacent_terms),
        "challenge_terms": list(COMMON_CHALLENGE_TERMS),
        "risk_terms": list(COMMON_RISK_TERMS),
        "opportunity_terms": list(COMMON_OPPORTUNITY_TERMS),
        "query_lenses": query_lenses,
        "signal_taxonomy": {k: list(v) for k, v in DEFAULT_SIGNAL_TAXONOMY.items()},
        "plan_source": "deterministic_fallback_no_llm",
    }


plan = fallback_signal_plan(INDUSTRY)
INDUSTRY_TERMS = plan["industry_terms"]
ADJACENT_TERMS = plan["adjacent_terms"]
CHALLENGE_TERMS = plan["challenge_terms"]
RISK_TERMS = plan["risk_terms"]
OPPORTUNITY_TERMS = plan["opportunity_terms"]
QUERY_LENSES = plan["query_lenses"]
SIGNAL_TAXONOMY = plan["signal_taxonomy"]

print("Plan source:", plan["plan_source"])
print("Industry terms:", INDUSTRY_TERMS[:10])
print("Query lenses:", [q["name"] for q in QUERY_LENSES])

Plan source: deterministic_fallback_no_llm
Industry terms: ['marketing agency', 'advertising agency', 'media agency', 'creative agency', 'performance marketing agency', 'digital agency', 'ad agency', 'agency holding company', 'adtech', 'martech']
Query lenses: ['core_industry', 'ai_automation', 'budget_demand', 'policy_regulation', 'platform_changes', 'competition_ma', 'labor_margin', 'technical_adoption']


In [30]:
def q(*parts: str) -> str:
    return " ".join(str(p).strip() for p in parts if str(p).strip())


if RUN_MODE == "measurement":
    ENABLED_SOURCES = list(MEASUREMENT_SOURCES)
else:
    ENABLED_SOURCES = list(MEASUREMENT_SOURCES) + list(DISCOVERY_EXTRA_SOURCES)

if not INCLUDE_WEBSITE_SOURCES:
    ENABLED_SOURCES = [s for s in ENABLED_SOURCES if s not in NON_API_WEBSITE_SOURCES]
ENABLED_SOURCES = [s for s in ENABLED_SOURCES if s not in EXCLUDED_SOURCES]

print("Enabled sources:", ", ".join(ENABLED_SOURCES))
print("Excluded sources:", ", ".join(sorted(EXCLUDED_SOURCES)))

Enabled sources: newsapi, guardian, edgar, federalregister, reddit, hackernews, openalex, crossref, openlibrary
Excluded sources: github, stackexchange, wikipedia


In [31]:
from typing import Literal, get_args, get_origin

from data_ingestion.config import FetcherSpec


def allowed_fetcher_sources() -> set[str] | None:
    try:
        annotation = FetcherSpec.model_fields["source"].annotation
        if get_origin(annotation) is Literal:
            return set(str(x) for x in get_args(annotation))
    except Exception:
        return None
    return None


available_sources = allowed_fetcher_sources()
api_key_status = pl.DataFrame(
    [
        {"env_var": env_var, "available": bool(os.getenv(env_var)), "used_by": source}
        for source, env_var in API_KEY_ENV_BY_SOURCE.items()
    ]
)

display(api_key_status)

if available_sources:
    missing_in_package = sorted(set(ENABLED_SOURCES) - available_sources)
    if missing_in_package:
        print(
            "Enabled sources not supported by the installed text-ingest package:",
            missing_in_package,
        )
else:
    print(
        "Could not inspect allowed FetcherSpec sources; proceeding with "
        "configured sources."
    )

env_var,available,used_by
str,bool,str
"""NEWSAPI_KEY""",false,"""newsapi"""
"""GUARDIAN_API_KEY""",false,"""guardian"""


In [32]:
def drop_none(d: dict[str, Any]) -> dict[str, Any]:
    return {k: v for k, v in d.items() if v is not None}


def base_config(query: str, **kwargs: Any) -> dict[str, Any]:
    return drop_none(
        {
            "query": query,
            "max_pages": MAX_PAGES,
            "start_date": START_DATE,
            "end_date": END_DATE,
            **BASE_FETCHER_CONFIG,
            **kwargs,
        }
    )


def source_config(source: str, query: str) -> dict[str, Any]:
    if source not in SOURCE_FETCHER_KWARGS:
        raise ValueError(f"No source_config implemented for {source}")
    kwargs = dict(SOURCE_FETCHER_KWARGS[source])
    api_key_env = API_KEY_ENV_BY_SOURCE.get(source)
    if api_key_env:
        kwargs["api_key"] = os.getenv(api_key_env)
    return base_config(query, **kwargs)


def should_skip_source(source: str) -> tuple[bool, str]:
    if source in EXCLUDED_SOURCES:
        return True, "explicitly excluded"
    if source in SOURCE_ENABLED_OVERRIDES and not SOURCE_ENABLED_OVERRIDES[source]:
        env_name = f"ENABLE_{source.upper()}_SOURCE"
        return True, f"disabled by default; set {env_name}=true to enable"
    if available_sources is not None and source not in available_sources:
        return True, "not supported by imported package"
    api_key_env = API_KEY_ENV_BY_SOURCE.get(source)
    if (
        source in HARD_SKIP_WHEN_MISSING_API_KEY
        and api_key_env
        and not os.getenv(api_key_env)
    ):
        return True, f"missing {api_key_env}"
    return False, ""


FETCHER_SPECS: list[dict[str, Any]] = []
FETCHER_SPEC_ROWS: list[dict[str, Any]] = []

lenses_df = pl.DataFrame(QUERY_LENSES).sort(["priority", "name"])
query_lenses_to_use = lenses_df.head(MAX_QUERY_VARIANTS).to_dicts()


def query_variants(lens: dict[str, Any]) -> list[str]:
    query = str(lens.get("query", "")).strip()
    variants = [query]
    if INDUSTRY.lower() not in query.lower():
        variants.append(q(INDUSTRY, query))
    if any(term in INDUSTRY.lower() for term in MARKETING_INDUSTRY_DETECT_TERMS):
        variants.append(q(MARKETING_QUERY_ANCHOR, query))

    seen = set()
    out = []
    for value in variants:
        value = re.sub(SPACE_REGEX, " ", value).strip()
        key = value.lower()
        if value and key not in seen:
            out.append(value)
            seen.add(key)
    return out[
        : QUERY_VARIANT_LIMIT_BY_MODE.get(
            RUN_MODE, QUERY_VARIANT_LIMIT_BY_MODE["discovery"]
        )
    ]


def query_variants_for_source(source: str, lens: dict[str, Any]) -> list[str]:
    """Return query variants adapted to source-specific API constraints."""
    return query_variants(lens)


idx = 0
specs_used_by_source: Counter[str] = Counter()
seen_fetch_cache_keys: set[str] = set()
for source in ENABLED_SOURCES:
    source_skip, source_skip_reason = should_skip_source(source)
    for lens in query_lenses_to_use:
        for query in query_variants_for_source(source, lens):
            idx += 1
            skip = source_skip
            reason = source_skip_reason

            spec_limit = SOURCE_SPEC_LIMITS.get(source)
            if (
                not skip
                and spec_limit is not None
                and specs_used_by_source[source] >= spec_limit
            ):
                skip = True
                limit_env = f"{source.upper()}_MAX_SPECS"
                reason = (
                    f"source spec cap reached ({spec_limit}); "
                    f"adjust {limit_env} to increase"
                )

            config = source_config(source, query) if not skip else {}
            cache_key = make_fetch_cache_key(source, config) if not skip else ""
            cache_file = fetch_cache_path(source, cache_key) if cache_key else None

            if (
                not skip
                and DEDUPLICATE_IDENTICAL_FETCH_SPECS
                and cache_key in seen_fetch_cache_keys
            ):
                skip = True
                reason = (
                    "duplicate equivalent fetch spec; prior spec has same "
                    "source/query/date/config"
                )

            row = {
                "spec_idx": idx,
                "source": source,
                "source_family": SOURCE_FAMILY.get(source, "other"),
                "lens": lens.get("name"),
                "priority": lens.get("priority"),
                "query": query,
                "enabled": not skip,
                "skip_reason": reason,
                "rate_limit_sensitive": source in RATE_LIMIT_SENSITIVE_SOURCES,
                "source_spec_limit": spec_limit,
                "fetch_cache_key": cache_key,
                "fetch_cache_file": str(cache_file) if cache_file else "",
                "fetch_cache_exists": bool(cache_file and cache_file.exists()),
            }
            FETCHER_SPEC_ROWS.append(row)
            if not skip:
                seen_fetch_cache_keys.add(cache_key)
                specs_used_by_source[source] += 1
                FETCHER_SPECS.append({"source": source, "config": config, "_meta": row})

fetcher_specs_df = rows_to_polars(FETCHER_SPEC_ROWS)
fetcher_specs_df.write_csv(FETCHER_SPECS_CSV)
print(f"Prepared {len(FETCHER_SPECS)} enabled specs; wrote {FETCHER_SPECS_CSV}")
display(
    fetcher_specs_df.group_by(["source_family", "source", "enabled", "skip_reason"])
    .len(name="specs")
    .sort(["source_family", "source"])
)

Prepared 56 enabled specs; wrote data/industry_signal_intelligence/runs/dt=2026-05-02/run_id=2026-05-02_20260503T012200Z/fetcher_specs.csv


source_family,source,enabled,skip_reason,specs
str,str,bool,str,u32
"""community""","""hackernews""",true,"""""",8
"""community""","""reddit""",true,"""""",8
"""company_filings""","""edgar""",true,"""""",8
"""news""","""guardian""",false,"""missing GUARDIAN_API_KEY""",8
"""news""","""newsapi""",false,"""missing NEWSAPI_KEY""",8
"""reference""","""openlibrary""",true,"""""",8
"""regulatory""","""federalregister""",true,"""""",8
"""research""","""crossref""",true,"""""",8
"""research""","""openalex""",true,"""""",8


In [33]:
import multiprocessing as mp
import queue as queue_module
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

from data_ingestion.config import JsonlSinkConfig, PipelineConfig
from data_ingestion.factories import build_fetcher
from data_ingestion.pipeline import DataDumperPipeline
from data_ingestion.sinks.jsonl import JsonlSink

try:
    from data_ingestion.config import RuntimeOptimizationConfig
except Exception:
    RuntimeOptimizationConfig = None


cache_write_lock = Lock()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    if not path.exists():
        return rows
    with path.open("r", encoding=JSON_ENCODING) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def write_jsonl(
    path: Path, rows: collections.abc.collections.abc.Iterable[dict[str, Any]]
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding=JSON_ENCODING) as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def safe_read_manifest(path: Path) -> pl.DataFrame:
    return safe_read_csv_polars(path)


def update_fetch_cache_manifest_rows(rows: list[dict[str, Any]]) -> None:
    """Merge new fetch-cache metadata into the persistent manifest once per run.

    Manifest writes are intentionally done in the main process after workers finish.
    This avoids multiple ingestion workers racing to update the same CSV file.
    """
    if not rows:
        return
    manifest = safe_read_manifest(FETCH_CACHE_MANIFEST_CSV)
    new_df = rows_to_polars(rows)
    if not manifest.is_empty() and "fetch_cache_key" in manifest.columns:
        new_keys = (
            [str(x) for x in new_df.get_column("fetch_cache_key").to_list()]
            if "fetch_cache_key" in new_df.columns
            else []
        )
        if new_keys:
            manifest = manifest.filter(
                ~pl.col("fetch_cache_key").cast(pl.Utf8).is_in(new_keys)
            )
    manifest = concat_polars([manifest, new_df])
    FETCH_CACHE_MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    manifest.write_csv(FETCH_CACHE_MANIFEST_CSV)


def make_pipeline_config() -> PipelineConfig:
    if RuntimeOptimizationConfig is None:
        return PipelineConfig(fail_fast=False)
    try:
        return PipelineConfig(
            fail_fast=False,
            runtime=RuntimeOptimizationConfig(**PIPELINE_RUNTIME_OPTIONS),
        )
    except Exception:
        return PipelineConfig(fail_fast=False)


def source_api_key_present(source: str) -> bool | None:
    api_key_env = API_KEY_ENV_BY_SOURCE.get(source)
    return bool(os.getenv(api_key_env)) if api_key_env else None


def rows_with_run_metadata(
    rows: list[dict[str, Any]],
    meta: dict[str, Any],
    source: str,
    cache_key: str,
    cache_hit: bool,
) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    for r in rows:
        rr = dict(r)
        rr.update(
            {
                "_run_id": RUN_ID,
                "_run_date": RUN_DATE,
                "_industry": INDUSTRY,
                "_spec_idx": int(meta["spec_idx"]),
                "_query": meta.get("query"),
                "_query_lens": meta.get("lens"),
                "_source_family": meta.get("source_family"),
                "_source_weight": SOURCE_WEIGHT.get(source, DEFAULT_SOURCE_WEIGHT),
                "_fetch_cache_key": cache_key,
                "_fetch_cache_hit": bool(cache_hit),
            }
        )
        out.append(rr)
    return out


def spec_paths_and_keys(
    spec: dict[str, Any],
) -> tuple[dict[str, Any], str, str, int, Path, str, Path]:
    meta = spec["_meta"].copy()
    source = spec["source"]
    lens = str(meta.get("lens", UNKNOWN_VALUE))
    spec_idx = int(meta["spec_idx"])
    spec_file = RAW_SPEC_DIR / SPEC_FILENAME_TEMPLATE.format(
        spec_idx=spec_idx,
        source=source,
        lens_slug=slugify(lens),
    )
    cache_key = str(
        meta.get("fetch_cache_key") or make_fetch_cache_key(source, spec["config"])
    )
    cache_file = fetch_cache_path(source, cache_key)
    return meta, source, lens, spec_idx, spec_file, cache_key, cache_file


def make_guard_result(
    spec: dict[str, Any],
    status: str,
    reason: str,
    started: float | None = None,
) -> dict[str, Any]:
    """Create a coverage-compatible result for failed/timeout specs."""
    meta, source, lens, spec_idx, spec_file, cache_key, cache_file = (
        spec_paths_and_keys(spec)
    )
    runtime_seconds = round(time.time() - started, 2) if started else np.nan
    return {
        "spec_idx": spec_idx,
        "source": source,
        "lens": lens,
        "status": status,
        "raw_rows": 0,
        "cache_hit": False,
        "coverage_row": {
            **meta,
            "status": status,
            "reason": reason,
            "raw_rows": 0,
            "raw_file": str(spec_file),
            "fetch_cache_key": cache_key,
            "fetch_cache_file": str(cache_file),
            "fetch_cache_hit": False,
            "runtime_seconds": runtime_seconds,
            "api_key_present": source_api_key_present(source),
        },
        "combined_rows": [],
        "manifest_row": None,
    }


def ingest_one_spec(spec: dict[str, Any]) -> dict[str, Any]:
    """Fetch one source/query spec, using the persistent raw cache when available."""
    meta, source, lens, spec_idx, spec_file, cache_key, cache_file = (
        spec_paths_and_keys(spec)
    )
    started = time.time()
    status = FETCH_CACHE_STATUS_MISS
    reason = ""
    raw_rows = 0
    cache_hit = False
    manifest_row: dict[str, Any] | None = None
    combined_spec_rows: list[dict[str, Any]] = []

    try:
        cached_rows: list[dict[str, Any]] = []
        if ENABLE_FETCH_CACHE and not FORCE_REFETCH and cache_file.exists():
            cached_rows = read_jsonl(cache_file)
            if cached_rows or REUSE_EMPTY_FETCH_CACHE:
                write_jsonl(spec_file, cached_rows)
                raw_rows = len(cached_rows)
                status = FETCH_CACHE_STATUS_HIT
                reason = "reused persistent fetch cache"
                cache_hit = True
            else:
                reason = "empty cache ignored; refetched"

        if not cache_hit:
            fetcher = build_fetcher({"source": source, "config": spec["config"]})
            sink = JsonlSink(JsonlSinkConfig(output_file=str(spec_file), append=False))
            pipeline = DataDumperPipeline(sink=sink, config=make_pipeline_config())
            _summary = pipeline.run([fetcher])
            fetched_rows = read_jsonl(spec_file)
            raw_rows = len(fetched_rows)
            status = FETCH_CACHE_STATUS_MISS
            if raw_rows == 0 and source in RATE_LIMIT_SENSITIVE_SOURCES:
                status = "empty_or_failed_quota_sensitive"
                reason = (
                    "zero rows from quota-sensitive source; check rate limit/quota logs"
                )

            if ENABLE_FETCH_CACHE and (fetched_rows or REUSE_EMPTY_FETCH_CACHE):
                with cache_write_lock:
                    write_jsonl(cache_file, fetched_rows)
                manifest_row = {
                    "fetch_cache_key": cache_key,
                    "source": source,
                    "source_family": meta.get("source_family"),
                    "lens": meta.get("lens"),
                    "query": meta.get("query"),
                    "start_date": START_DATE,
                    "end_date": END_DATE,
                    "max_pages": MAX_PAGES,
                    "raw_rows": raw_rows,
                    "cache_file": str(cache_file),
                    "created_at_utc": datetime.now(timezone.utc).isoformat(),
                    "schema_version": FETCH_CACHE_SCHEMA_VERSION,
                    "sanitized_config_json": json.dumps(
                        sanitize_for_cache(spec["config"]),
                        sort_keys=True,
                        ensure_ascii=False,
                        default=str,
                    ),
                }

        rows = read_jsonl(spec_file)
        combined_spec_rows = rows_with_run_metadata(
            rows, meta, source, cache_key, cache_hit
        )

    except Exception as exc:
        status = FETCH_CACHE_STATUS_FAILED
        reason = f"{type(exc).__name__}: {exc}"

    coverage_row = {
        **meta,
        "status": status,
        "reason": reason,
        "raw_rows": raw_rows,
        "raw_file": str(spec_file),
        "fetch_cache_key": cache_key,
        "fetch_cache_file": str(cache_file),
        "fetch_cache_hit": cache_hit,
        "runtime_seconds": round(time.time() - started, 2),
        "api_key_present": source_api_key_present(source),
    }

    return {
        "spec_idx": spec_idx,
        "source": source,
        "lens": lens,
        "status": status,
        "raw_rows": raw_rows,
        "cache_hit": cache_hit,
        "coverage_row": coverage_row,
        "combined_rows": combined_spec_rows,
        "manifest_row": manifest_row,
    }


def _ingest_process_target(spec: dict[str, Any], result_queue: Any) -> None:
    """Child-process wrapper used by hard-timeout ingestion.

    Important: do NOT send fetched records through multiprocessing.Queue.
    Large EDGAR/news pulls can block while serializing/flushing the queue, making
    the parent think the fetch is stuck even after JSONL was written. The child
    only returns small metadata; the parent re-reads the spec JSONL file.
    """
    try:
        result = ingest_one_spec(spec)
        deferred_count = len(result.get("combined_rows") or [])
        result["combined_rows"] = []
        result["_combined_rows_deferred"] = True
        result["_deferred_combined_row_count"] = deferred_count
        result_queue.put({"ok": True, "result": result})
    except BaseException as exc:
        with contextlib.suppress(Exception):
            result_queue.put(
                {
                    "ok": False,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )


def get_mp_context() -> Any:
    try:
        return mp.get_context(INGESTION_MULTIPROCESS_START_METHOD)
    except Exception:
        return mp.get_context("fork")


def start_ingest_process(spec: dict[str, Any], ctx: Any) -> dict[str, Any]:
    q = ctx.Queue(maxsize=1)
    p = ctx.Process(target=_ingest_process_target, args=(spec, q), daemon=True)
    started = time.time()
    p.start()
    return {"process": p, "queue": q, "spec": spec, "started": started}


def finalize_deferred_process_result(
    result: dict[str, Any], spec: dict[str, Any]
) -> dict[str, Any]:
    """Rehydrate fetched rows in the parent process after child fetch completion."""
    if not result.get("_combined_rows_deferred"):
        return result

    meta, source, _lens, _spec_idx, spec_file, cache_key, _cache_file = (
        spec_paths_and_keys(spec)
    )
    coverage_row = result.get("coverage_row") or {}
    raw_file = Path(str(coverage_row.get("raw_file") or spec_file))
    cache_hit = bool(result.get("cache_hit") or coverage_row.get("fetch_cache_hit"))
    cache_key_for_rows = str(coverage_row.get("fetch_cache_key") or cache_key)

    rows = read_jsonl(raw_file)
    combined_rows = rows_with_run_metadata(
        rows, meta, source, cache_key_for_rows, cache_hit
    )
    result["combined_rows"] = combined_rows

    result["raw_rows"] = len(rows)
    if isinstance(result.get("coverage_row"), dict):
        result["coverage_row"]["raw_rows"] = len(rows)
        if (
            len(rows) > 0
            and result["coverage_row"].get("status") == FETCH_CACHE_STATUS_TIMEOUT
        ):
            result["coverage_row"]["status"] = result.get(
                "status", FETCH_CACHE_STATUS_MISS
            )
    return result


def collect_process_result(info: dict[str, Any]) -> dict[str, Any]:
    spec = info["spec"]
    q = info["queue"]
    try:
        payload = q.get(timeout=INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS)
    except queue_module.Empty:
        reason = (
            "process exited but result metadata was not available within "
            "INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS="
            f"{INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS}"
        )
        return make_guard_result(
            spec,
            FETCH_CACHE_STATUS_FAILED,
            reason,
            info["started"],
        )
    if payload.get("ok"):
        return finalize_deferred_process_result(payload["result"], spec)
    return make_guard_result(
        spec,
        FETCH_CACHE_STATUS_FAILED,
        f"{payload.get('error_type')}: {payload.get('error')}",
        info["started"],
    )


def run_one_spec_with_hard_timeout(spec: dict[str, Any]) -> dict[str, Any]:
    """Run one spec in a child process and terminate it if it exceeds the timeout."""
    ctx = get_mp_context()
    info = start_ingest_process(spec, ctx)
    p = info["process"]
    p.join(INGESTION_SPEC_TIMEOUT_SECONDS)
    if p.is_alive():
        p.terminate()
        p.join(5)
        if p.is_alive():
            try:
                p.kill()
                p.join(5)
            except Exception:
                pass
        return make_guard_result(
            spec,
            FETCH_CACHE_STATUS_TIMEOUT,
            (
                "terminated after INGESTION_SPEC_TIMEOUT_SECONDS="
                f"{INGESTION_SPEC_TIMEOUT_SECONDS}"
            ),
            info["started"],
        )
    return collect_process_result(info)


def log_ingestion_completion(
    result: dict[str, Any],
    completed: int,
    total: int,
    started_at: float,
    rows_so_far: int,
    cache_hit_count: int,
    api_fetch_count: int,
    failed_count: int,
    timeout_count: int,
    remaining_sources: int,
) -> None:
    if not ENABLE_PROGRESS_LOGGING:
        return
    if (
        completed != total
        and INGESTION_LOG_EVERY_SPECS > 0
        and completed % INGESTION_LOG_EVERY_SPECS != 0
    ):
        return
    progress_log(
        "ingestion:done",
        completed,
        total,
        current_source=result.get("source"),
        current_lens=slugify(result.get("lens"), 36),
        status=result.get("status"),
        last_rows=f"{int(result.get('raw_rows') or 0):,}",
        rows_so_far=f"{rows_so_far:,}",
        cache_hits=cache_hit_count,
        api_fetches=api_fetch_count,
        failures=failed_count,
        timeouts=timeout_count,
        sources_remaining=remaining_sources,
        elapsed=format_duration(time.time() - started_at),
        eta=format_duration(estimated_remaining_seconds(started_at, completed, total)),
    )


def update_ingestion_accumulators(
    result: dict[str, Any],
    completed: int,
    total_specs: int,
    started_at: float,
    remaining_sources: int,
) -> None:
    global rows_so_far, cache_hit_count, api_fetch_count, failed_count, timeout_count
    coverage_rows.append(result["coverage_row"])
    combined_rows.extend(result["combined_rows"])
    if result.get("manifest_row"):
        manifest_rows_to_write.append(result["manifest_row"])

    rows_so_far += int(result.get("raw_rows") or 0)
    cache_hit_count += int(bool(result.get("cache_hit")))
    failed_count += int(result.get("status") == FETCH_CACHE_STATUS_FAILED)
    timeout_count += int(result.get("status") == FETCH_CACHE_STATUS_TIMEOUT)
    api_fetch_count += int(
        (not result.get("cache_hit"))
        and result.get("status")
        not in {FETCH_CACHE_STATUS_FAILED, FETCH_CACHE_STATUS_TIMEOUT}
    )

    log_ingestion_completion(
        result,
        completed,
        total_specs,
        started_at,
        rows_so_far,
        cache_hit_count,
        api_fetch_count,
        failed_count,
        timeout_count,
        remaining_sources,
    )


coverage_rows: list[dict[str, Any]] = []
combined_rows: list[dict[str, Any]] = []
manifest_rows_to_write: list[dict[str, Any]] = []

total_specs = len(FETCHER_SPECS)
ingestion_started = time.time()
api_fetch_count = 0
cache_hit_count = 0
failed_count = 0
timeout_count = 0
rows_so_far = 0
unique_sources_total = len({spec["source"] for spec in FETCHER_SPECS})
use_parallel = (
    ENABLE_PARALLEL_INGESTION and MAX_INGESTION_WORKERS > 1 and total_specs > 1
)
worker_count = min(MAX_INGESTION_WORKERS, total_specs) if use_parallel else 1
use_hard_timeout = ENABLE_HARD_FETCH_TIMEOUT
if (
    use_hard_timeout
    and os.name == "nt"
    and INGESTION_MULTIPROCESS_START_METHOD == "fork"
):
    print(
        "Hard fetch timeout uses multiprocessing. On Windows, set "
        "INGESTION_MULTIPROCESS_START_METHOD=spawn or disable "
        "ENABLE_HARD_FETCH_TIMEOUT."
    )

if ENABLE_PROGRESS_LOGGING:
    print(
        f"Starting ingestion: specs={total_specs:,}, "
        f"unique_sources={unique_sources_total:,}, "
        f"date_window={START_DATE}..{END_DATE}, max_pages={MAX_PAGES}, "
        f"fetch_cache={ENABLE_FETCH_CACHE}, force_refetch={FORCE_REFETCH}, "
        f"parallel={use_parallel}, workers={worker_count}, "
        f"hard_timeout={use_hard_timeout}, "
        f"per_spec_timeout_seconds={INGESTION_SPEC_TIMEOUT_SECONDS}, "
        f"result_queue_timeout_seconds={INGESTION_RESULT_QUEUE_TIMEOUT_SECONDS}, "
        f"serial_sources={','.join(sorted(SERIAL_INGESTION_SOURCES)) or 'none'}, "
        f"dedupe_identical_specs={DEDUPLICATE_IDENTICAL_FETCH_SPECS}"
    )
    removed_sources_enabled = sorted(set(ENABLED_SOURCES) & REMOVED_SOURCES)
    if removed_sources_enabled:
        print(
            "Removed sources still appeared in ENABLED_SOURCES and will "
            f"be excluded: {removed_sources_enabled}"
        )

if not use_parallel:
    for i, spec in enumerate(FETCHER_SPECS, start=1):
        meta = spec["_meta"].copy()
        source = spec["source"]
        lens = str(meta.get("lens", UNKNOWN_VALUE))
        specs_remaining_before = total_specs - i + 1
        sources_remaining_after_this = len({s["source"] for s in FETCHER_SPECS[i:]})
        should_log_start = (i == 1) or (
            INGESTION_LOG_EVERY_SPECS > 0 and (i - 1) % INGESTION_LOG_EVERY_SPECS == 0
        )
        if should_log_start:
            progress_log(
                "ingestion:start",
                i - 1,
                total_specs,
                current_source=source,
                current_lens=slugify(lens, 36),
                specs_left_including_current=specs_remaining_before,
                sources_left_after_current=sources_remaining_after_this,
                rows_so_far=f"{rows_so_far:,}",
                cache_hits=cache_hit_count,
                api_fetches=api_fetch_count,
                failures=failed_count,
                timeouts=timeout_count,
                eta=format_duration(
                    estimated_remaining_seconds(ingestion_started, i - 1, total_specs)
                ),
            )

        result = (
            run_one_spec_with_hard_timeout(spec)
            if use_hard_timeout
            else ingest_one_spec(spec)
        )
        update_ingestion_accumulators(
            result,
            i,
            total_specs,
            ingestion_started,
            sources_remaining_after_this,
        )
elif use_hard_timeout:
    ctx = get_mp_context()
    pending_specs = list(FETCHER_SPECS)
    running: dict[int, dict[str, Any]] = {}
    completed = 0
    completed_sources: Counter[str] = Counter()
    total_by_source: Counter[str] = Counter(spec["source"] for spec in FETCHER_SPECS)

    def launch_next_specs() -> None:
        while pending_specs and len(running) < worker_count:
            running_sources = {
                str(info["spec"].get("source")) for info in running.values()
            }
            launch_idx = None
            for candidate_idx, candidate_spec in enumerate(pending_specs):
                candidate_source = str(candidate_spec.get("source"))
                if (
                    candidate_source in SERIAL_INGESTION_SOURCES
                    and candidate_source in running_sources
                ):
                    continue
                launch_idx = candidate_idx
                break
            if launch_idx is None:
                return

            spec_to_launch = pending_specs.pop(launch_idx)
            info = start_ingest_process(spec_to_launch, ctx)
            p = info["process"]
            running[p.pid] = info
            if ENABLE_PROGRESS_LOGGING:
                meta = spec_to_launch["_meta"]
                progress_log(
                    "ingestion:start",
                    completed,
                    total_specs,
                    current_source=spec_to_launch["source"],
                    current_lens=slugify(meta.get("lens", UNKNOWN_VALUE), 36),
                    running_workers=len(running),
                    queued_specs=len(pending_specs),
                    serial_sources=",".join(sorted(SERIAL_INGESTION_SOURCES)) or "none",
                    rows_so_far=f"{rows_so_far:,}",
                    failures=failed_count,
                    timeouts=timeout_count,
                    eta=format_duration(
                        estimated_remaining_seconds(
                            ingestion_started, completed, total_specs
                        )
                    ),
                )

    launch_next_specs()
    while running:
        made_progress = False
        now = time.time()
        for pid, info in list(running.items()):
            p = info["process"]
            spec = info["spec"]
            elapsed = now - info["started"]
            if not p.is_alive():
                p.join(1)
                result = collect_process_result(info)
            elif elapsed > INGESTION_SPEC_TIMEOUT_SECONDS:
                p.terminate()
                p.join(5)
                if p.is_alive():
                    try:
                        p.kill()
                        p.join(5)
                    except Exception:
                        pass
                result = make_guard_result(
                    spec,
                    FETCH_CACHE_STATUS_TIMEOUT,
                    (
                        "terminated after INGESTION_SPEC_TIMEOUT_SECONDS="
                        f"{INGESTION_SPEC_TIMEOUT_SECONDS}"
                    ),
                    info["started"],
                )
            else:
                continue

            del running[pid]
            completed += 1
            completed_sources[str(result.get("source"))] += 1
            remaining_sources = sum(
                1
                for src, total in total_by_source.items()
                if completed_sources[src] < total
            )
            update_ingestion_accumulators(
                result,
                completed,
                total_specs,
                ingestion_started,
                remaining_sources,
            )
            made_progress = True

        launch_next_specs()
        if running and not made_progress:
            time.sleep(INGESTION_POLL_SECONDS)
else:
    futures = {}
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        for spec in FETCHER_SPECS:
            fut = executor.submit(ingest_one_spec, spec)
            futures[fut] = spec

        completed = 0
        completed_sources: Counter[str] = Counter()
        total_by_source: Counter[str] = Counter(
            spec["source"] for spec in FETCHER_SPECS
        )

        for completed, fut in enumerate(as_completed(futures), start=1):
            spec = futures[fut]
            try:
                result = fut.result()
            except Exception as exc:
                result = make_guard_result(
                    spec, FETCH_CACHE_STATUS_FAILED, f"{type(exc).__name__}: {exc}"
                )

            completed_sources[str(result.get("source"))] += 1
            remaining_sources = sum(
                1
                for src, total in total_by_source.items()
                if completed_sources[src] < total
            )
            update_ingestion_accumulators(
                result,
                completed,
                total_specs,
                ingestion_started,
                remaining_sources,
            )


if ENABLE_FETCH_CACHE:
    update_fetch_cache_manifest_rows(manifest_rows_to_write)


coverage_rows = sorted(coverage_rows, key=lambda r: int(r.get("spec_idx") or 0))
combined_rows = sorted(
    combined_rows,
    key=lambda r: (
        int(r.get("_spec_idx") or 0),
        str(r.get("url") or r.get("external_id") or r.get("title") or ""),
    ),
)

write_jsonl(COMBINED_RAW_JSONL, combined_rows)
coverage_df = rows_to_polars(coverage_rows)
if not coverage_df.is_empty() and "spec_idx" in coverage_df.columns:
    coverage_df = coverage_df.sort("spec_idx")
coverage_df.write_csv(COVERAGE_CSV)

print(f"Combined raw rows: {len(combined_rows):,}")
print(f"Wrote combined raw JSONL: {COMBINED_RAW_JSONL}")
print(f"Wrote coverage CSV: {COVERAGE_CSV}")
if ENABLE_FETCH_CACHE:
    cache_hits_total = (
        int(coverage_df.select(pl.col("fetch_cache_hit").fill_null(False).sum()).item())
        if "fetch_cache_hit" in coverage_df.columns and not coverage_df.is_empty()
        else 0
    )
    print(f"Fetch cache manifest: {FETCH_CACHE_MANIFEST_CSV}")
    print(f"Cache hits: {cache_hits_total:,} / {len(coverage_df):,} specs")
print(
    f"Ingestion elapsed: {format_duration(time.time() - ingestion_started)} | "
    f"parallel={use_parallel} | workers={worker_count} | "
    f"hard_timeout={use_hard_timeout} | "
    f"API fetches: {api_fetch_count:,} | failures: {failed_count:,} | "
    f"timeouts: {timeout_count:,}"
)

coverage_summary_display = (
    (
        coverage_df.group_by(["source_family", "source", "status"])
        .agg(
            [
                pl.col("spec_idx").count().alias("specs"),
                pl.col("raw_rows").sum().alias("raw_rows"),
                pl.col("fetch_cache_hit").cast(pl.Int64).sum().alias("cache_hits"),
                pl.col("reason").drop_nulls().first().alias("reason"),
            ]
        )
        .sort(["source_family", "source", "status"])
    )
    if not coverage_df.is_empty()
    else pl_empty()
)
display(coverage_summary_display)

if timeout_count:
    print(
        "WARNING: Some fetch specs timed out and were skipped. "
        "Inspect coverage.csv for status='timeout'. Increase "
        "INGESTION_SPEC_TIMEOUT_SECONDS if the source is slow but useful."
    )
if len(combined_rows) < LOW_VOLUME_WARNING_THRESHOLD:
    print(
        "WARNING: This is a low-volume run. Check missing API keys, "
        "source failures, query narrowness, or increase "
        "MAX_PAGES/MAX_QUERY_VARIANTS."
    )
if not combined_rows:
    raise RuntimeError(
        "No records ingested. Check API keys, enabled sources, fetcher "
        "support, or query plan."
    )

Starting ingestion: specs=56, unique_sources=7, date_window=2025-11-03..2026-05-02, max_pages=10, fetch_cache=True, force_refetch=False, parallel=True, workers=2, hard_timeout=True, per_spec_timeout_seconds=180, result_queue_timeout_seconds=15, serial_sources=edgar,federalregister, dedupe_identical_specs=True
[ingestion:start] 0/56 (0.0%) | current_source=edgar | current_lens=ai_automation | running_workers=1 | queued_specs=55 | serial_sources=edgar,federalregister | rows_so_far=0 | failures=0 | timeouts=0 | eta=unknown
[ingestion:start] 0/56 (0.0%) | current_source=federalregister | current_lens=ai_automation | running_workers=2 | queued_specs=54 | serial_sources=edgar,federalregister | rows_so_far=0 | failures=0 | timeouts=0 | eta=unknown
[ingestion:done] 1/56 (1.8%) | current_source=edgar | current_lens=ai_automation | status=cache_hit | last_rows=1,000 | rows_so_far=1,000 | cache_hits=1 | api_fetches=0 | failures=0 | timeouts=0 | sources_remaining=7 | elapsed=2s | eta=unknown
[inge

2026-05-02 21:22:02,507 | INFO     | data_ingestion.factories | Built fetcher source=federalregister class=FederalRegisterFetcher
2026-05-02 21:22:02,511 | INFO     | data_ingestion.pipeline | Pipeline initialized fail_fast=False enrich_full_text=False sink_write_batch_size=500 transforms_enabled=False resume=False checkpoint=None
2026-05-02 21:22:02,514 | INFO     | data_ingestion.pipeline | Pipeline run started
2026-05-02 21:22:02,516 | INFO     | data_ingestion.pipeline | Starting source=federalregister
2026-05-02 21:22:02,518 | INFO     | data_ingestion.fetchers.federal_register | FederalRegister: requesting page=1 per_page=100 start_date=2025-11-03 end_date=2026-05-02
2026-05-02 21:22:02,621 | INFO     | data_ingestion.fetchers.federal_register | FederalRegister: no results page=1 pages_fetched=0
2026-05-02 21:22:02,623 | INFO     | data_ingestion.pipeline | Finished source=federalregister seen=0 kept=0 dropped_by_topic=0 dropped_by_transform=0
2026-05-02 21:22:02,626 | INFO     |

[ingestion:start] 2/56 (3.6%) | current_source=federalregister | current_lens=ai_automation | running_workers=2 | queued_specs=52 | serial_sources=edgar,federalregister | rows_so_far=1,003 | failures=0 | timeouts=0 | eta=58s
[ingestion:done] 3/56 (5.4%) | current_source=edgar | current_lens=ai_automation | status=cache_hit | last_rows=1,000 | rows_so_far=2,003 | cache_hits=3 | api_fetches=0 | failures=0 | timeouts=0 | sources_remaining=7 | elapsed=4s | eta=1m13s
[ingestion:done] 4/56 (7.1%) | current_source=federalregister | current_lens=ai_automation | status=fetched | last_rows=0 | rows_so_far=2,003 | cache_hits=3 | api_fetches=1 | failures=0 | timeouts=0 | sources_remaining=7 | elapsed=4s | eta=54s
[ingestion:start] 4/56 (7.1%) | current_source=edgar | current_lens=budget_demand | running_workers=1 | queued_specs=51 | serial_sources=edgar,federalregister | rows_so_far=2,003 | failures=0 | timeouts=0 | eta=54s
[ingestion:start] 4/56 (7.1%) | current_source=federalregister | current_l

2026-05-02 21:22:14,907 | INFO     | data_ingestion.factories | Built fetcher source=federalregister class=FederalRegisterFetcher
2026-05-02 21:22:14,913 | INFO     | data_ingestion.pipeline | Pipeline initialized fail_fast=False enrich_full_text=False sink_write_batch_size=500 transforms_enabled=False resume=False checkpoint=None
2026-05-02 21:22:14,915 | INFO     | data_ingestion.pipeline | Pipeline run started
2026-05-02 21:22:14,916 | INFO     | data_ingestion.pipeline | Starting source=federalregister
2026-05-02 21:22:14,918 | INFO     | data_ingestion.fetchers.federal_register | FederalRegister: requesting page=1 per_page=100 start_date=2025-11-03 end_date=2026-05-02


[ingestion:start] 14/56 (25.0%) | current_source=federalregister | current_lens=policy_regulation | running_workers=2 | queued_specs=40 | serial_sources=edgar,federalregister | rows_so_far=7,348 | failures=0 | timeouts=0 | eta=44s


2026-05-02 21:22:15,053 | INFO     | data_ingestion.fetchers.federal_register | FederalRegister: no results page=1 pages_fetched=0
2026-05-02 21:22:15,056 | INFO     | data_ingestion.pipeline | Finished source=federalregister seen=0 kept=0 dropped_by_topic=0 dropped_by_transform=0
2026-05-02 21:22:15,059 | INFO     | data_ingestion.pipeline | Pipeline run completed total_records=0 failed_sources=0 output=data/industry_signal_intelligence/runs/dt=2026-05-02/run_id=2026-05-02_20260503T012200Z/raw_by_spec/spec_0032_federalregister_policy_regulation.jsonl


[ingestion:done] 15/56 (26.8%) | current_source=edgar | current_lens=policy_regulation | status=cache_hit | last_rows=1,000 | rows_so_far=8,348 | cache_hits=14 | api_fetches=1 | failures=0 | timeouts=0 | sources_remaining=6 | elapsed=17s | eta=45s
[ingestion:done] 16/56 (28.6%) | current_source=federalregister | current_lens=policy_regulation | status=fetched | last_rows=0 | rows_so_far=8,348 | cache_hits=14 | api_fetches=2 | failures=0 | timeouts=0 | sources_remaining=5 | elapsed=17s | eta=41s
[ingestion:start] 16/56 (28.6%) | current_source=reddit | current_lens=ai_automation | running_workers=1 | queued_specs=39 | serial_sources=edgar,federalregister | rows_so_far=8,348 | failures=0 | timeouts=0 | eta=41s
[ingestion:start] 16/56 (28.6%) | current_source=reddit | current_lens=ai_automation | running_workers=2 | queued_specs=38 | serial_sources=edgar,federalregister | rows_so_far=8,348 | failures=0 | timeouts=0 | eta=41s
[ingestion:done] 17/56 (30.4%) | current_source=reddit | current

KeyboardInterrupt: 

In [ ]:
def norm_text(value: Any) -> str:
    if is_missing(value):
        return ""
    if isinstance(value, list):
        return " ".join(norm_text(v) for v in value)
    return str(value)


def compact_spaces(text: str) -> str:
    return re.sub(SPACE_REGEX, " ", str(text)).strip()


def normalize_title(title: str) -> str:
    title = str(title).lower()
    title = re.sub(URL_REGEX, " ", title)
    title = re.sub(NON_ALNUM_REGEX, " ", title)
    return compact_spaces(title)


def canonical_domain(url: str) -> str:
    try:
        host = urlparse(str(url)).netloc.lower()
        host = re.sub(WWW_PREFIX_REGEX, "", host)
        return host
    except Exception:
        return ""


def stable_hash(*parts: Any) -> str:
    text = HASH_JOINER.join(norm_text(p).lower().strip() for p in parts)
    return hashlib.sha256(text.encode(JSON_ENCODING)).hexdigest()[:HASH_LENGTH]


raw_rows = read_jsonl(COMBINED_RAW_JSONL)
end_date_obj = parse_date(END_DATE)
run_date_obj = parse_date(RUN_DATE)
processed_rows = []

for row in raw_rows:
    r = dict(row)
    for col in RAW_REQUIRED_COLUMNS:
        r.setdefault(col, None)

    r["title"] = compact_spaces(norm_text(r.get("title")))
    r["abstract"] = compact_spaces(norm_text(r.get("abstract")))
    r["full_text"] = compact_spaces(norm_text(r.get("full_text")))
    r["text"] = compact_spaces(f"{r['title']}\n{r['abstract']}\n{r['full_text']}")
    r["domain"] = canonical_domain(r.get("url", ""))
    r["normalized_title"] = normalize_title(r.get("title", ""))

    event_date = (
        parse_date(r.get("published_date"), default=None)
        if not is_missing(r.get("published_date"))
        else None
    )
    if event_date is None:
        event_date = (
            parse_date(r.get("fetched_at"), default=None)
            if not is_missing(r.get("fetched_at"))
            else None
        )
    if event_date is None:
        event_date = run_date_obj
    r["event_date"] = event_date.isoformat()
    r["age_days"] = max((end_date_obj - event_date).days, 0)

    r["record_fingerprint"] = stable_hash(
        r.get("domain"), r.get("normalized_title"), r.get("event_date")
    )
    r["record_id"] = stable_hash(
        r.get("source"), r.get("external_id"), r.get("url"), r.get("record_fingerprint")
    )

    source_weight = to_float(r.get("_source_weight"), DEFAULT_DEDUPE_SOURCE_WEIGHT)
    has_url = 0.0 if is_missing(r.get("url")) else 1.0
    text_len_component = (
        min(len(r.get("text", "")), FULL_TEXT_DEDUPE_CAP) / FULL_TEXT_DEDUPE_DIVISOR
    )
    r["_dedupe_rank"] = source_weight + has_url * DEDUPE_URL_BONUS + text_len_component
    processed_rows.append(r)

raw_df = rows_to_polars(processed_rows)
before = len(raw_df)
clean_df = (
    raw_df.sort("_dedupe_rank", descending=True)
    .unique(subset=["record_fingerprint"], keep="first")
    .drop("_dedupe_rank")
)
print(
    f"Rows before dedupe: {before:,}; after dedupe: {len(clean_df):,}; "
    f"duplicates removed: {before - len(clean_df):,}"
)
display(
    clean_df.select(
        ["source", "_source_family", "_query_lens", "event_date", "title", "url"]
    ).head(CLEAN_RECORDS_DISPLAY_ROWS)
)

In [ ]:
def term_hits(text: str, terms: list[str]) -> list[str]:
    text_l = f" {str(text).lower()} "
    hits = []
    for term in terms:
        term_l = str(term).lower().strip()
        if not term_l:
            continue
        if re.search(
            TERM_LEFT_BOUNDARY + re.escape(term_l) + TERM_RIGHT_BOUNDARY, text_l
        ):
            hits.append(term)
    return sorted(set(hits), key=lambda x: str(x).lower())


def categories_for_text(text: str, taxonomy: dict[str, list[str]]) -> list[str]:
    cats = []
    for cat, terms in taxonomy.items():
        if term_hits(text, terms):
            cats.append(cat)
    return cats or [UNCATEGORIZED_CATEGORY]


def primary_category(cats: list[str]) -> str:
    for p in CATEGORY_PRIORITY:
        if p in cats:
            return p
    return cats[0] if cats else UNCATEGORIZED_CATEGORY


def term_occurrences(text: str, terms: list[str]) -> list[tuple[str, int]]:
    text_l = str(text).lower()
    occurrences: list[tuple[str, int]] = []
    spans: list[tuple[int, int]] = []
    for term in sorted(
        {str(t).lower().strip() for t in terms if str(t).strip()}, key=len, reverse=True
    ):
        pattern = TERM_LEFT_BOUNDARY + re.escape(term) + TERM_RIGHT_BOUNDARY
        for match in re.finditer(pattern, text_l):
            span = match.span()
            if any(max(span[0], prev[0]) < min(span[1], prev[1]) for prev in spans):
                continue
            occurrences.append((term, match.start()))
            spans.append(span)
    return occurrences


def has_direction_negation(text_l: str, start: int) -> bool:
    left_context = text_l[max(0, start - DIRECTION_CONTEXT_CHARS) : start]
    return bool(term_hits(left_context, DIRECTION_NEGATION_TERMS))


def direction_evidence(text: str) -> dict[str, Any]:
    text_l = str(text).lower()
    risk_terms = RISK_TERMS + CHALLENGE_TERMS
    opportunity_terms = OPPORTUNITY_TERMS
    strong_risk = set(str(t).lower() for t in STRONG_RISK_TERMS)
    strong_opp = set(str(t).lower() for t in STRONG_OPPORTUNITY_TERMS)
    ambiguous_risk = set(str(t).lower() for t in AMBIGUOUS_RISK_TERMS)
    ambiguous_opp = set(str(t).lower() for t in AMBIGUOUS_OPPORTUNITY_TERMS)
    risk_score = 0.0
    opportunity_score = 0.0
    risk_hits: set[str] = set()
    opportunity_hits: set[str] = set()

    for term, start in term_occurrences(text_l, risk_terms):
        weight = 2.0 if term in strong_risk else 0.5 if term in ambiguous_risk else 1.0
        if has_direction_negation(text_l, start):
            opportunity_score += min(weight, 1.0)
        else:
            risk_score += weight
            risk_hits.add(term)

    for term, start in term_occurrences(text_l, opportunity_terms):
        weight = 2.0 if term in strong_opp else 0.5 if term in ambiguous_opp else 1.0
        if has_direction_negation(text_l, start):
            risk_score += min(weight, 1.0)
        else:
            opportunity_score += weight
            opportunity_hits.add(term)

    margin = abs(risk_score - opportunity_score)
    if (
        max(risk_score, opportunity_score) < DIRECTION_MIN_EVIDENCE_SCORE
        or margin < DIRECTION_RESOLUTION_MARGIN
    ):
        direction = "neutral"
    elif risk_score > opportunity_score:
        direction = "risk"
    else:
        direction = "opportunity"
    return {
        "direction": direction,
        "risk_score": round(risk_score, 3),
        "opportunity_score": round(opportunity_score, 3),
        "risk_hits": sorted(risk_hits),
        "opportunity_hits": sorted(opportunity_hits),
    }


def infer_direction(text: str) -> str:
    return direction_evidence(text)["direction"]


def minmax_values(values: collections.abc.Iterable[Any]) -> list[float]:
    arr = np.array([to_float(v, 0.0) for v in values], dtype=float)
    if len(arr) == 0 or np.nanmax(arr) == np.nanmin(arr):
        return [0.0 for _ in arr]
    return ((arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr))).tolist()


rows = clean_df.to_dicts()

if not rows:
    df = pl_empty()
    print("No clean records available for classification.")
else:
    relevance_raw = []
    signal_strength_raw = []
    for r in rows:
        text = r.get("text", "") or ""
        r["matched_industry_terms"] = term_hits(text, INDUSTRY_TERMS)
        r["matched_adjacent_terms"] = term_hits(text, ADJACENT_TERMS)
        r["matched_challenge_terms"] = term_hits(text, CHALLENGE_TERMS)
        r["matched_risk_terms"] = term_hits(text, RISK_TERMS)
        r["matched_opportunity_terms"] = term_hits(text, OPPORTUNITY_TERMS)
        r["signal_categories_rule"] = categories_for_text(text, SIGNAL_TAXONOMY)
        r["primary_category_rule"] = primary_category(r["signal_categories_rule"])
        direction_info = direction_evidence(text)
        r["direction_rule"] = direction_info["direction"]
        r["direction_risk_score"] = direction_info["risk_score"]
        r["direction_opportunity_score"] = direction_info["opportunity_score"]

        r["industry_term_count"] = len(r["matched_industry_terms"])
        r["adjacent_term_count"] = len(r["matched_adjacent_terms"])
        r["challenge_term_count"] = len(r["matched_challenge_terms"])
        r["risk_term_count"] = len(r["matched_risk_terms"])
        r["opportunity_term_count"] = len(r["matched_opportunity_terms"])
        r["is_relevant_heuristic"] = bool(
            (r["industry_term_count"] > 0)
            or (
                (r["adjacent_term_count"] > 0)
                and (r["primary_category_rule"] != UNCATEGORIZED_CATEGORY)
            )
        )

        r["signal_categories"] = r["signal_categories_rule"]
        r["primary_category"] = r["primary_category_rule"]
        r["direction"] = r["direction_rule"]
        r["freshness_score"] = math.exp(
            -max(to_float(r.get("age_days"), DEFAULT_MISSING_AGE_DAYS), 0.0)
            / FRESHNESS_HALFLIFE_DAYS
        )

        relevance_raw.append(
            r["industry_term_count"] * RULE_RELEVANCE_INDUSTRY_WEIGHT
            + r["adjacent_term_count"]
        )
        signal_strength_raw.append(
            r["challenge_term_count"]
            + r["risk_term_count"]
            + r["opportunity_term_count"]
        )

    relevance_scores = minmax_values(relevance_raw)
    signal_strength_scores = minmax_values(signal_strength_raw)
    for r, rel_score, sig_score in zip(
        rows, relevance_scores, signal_strength_scores, strict=True
    ):
        r["relevance_score"] = rel_score
        r["signal_strength_score"] = sig_score
        source_weight = to_float(r.get("_source_weight"), DEFAULT_SOURCE_WEIGHT)
        r["triage_score"] = round(
            TRIAGE_SCORE_SCALE
            * (
                RULE_TRIAGE_WEIGHTS["relevance"] * r["relevance_score"]
                + RULE_TRIAGE_WEIGHTS["freshness"] * r["freshness_score"]
                + RULE_TRIAGE_WEIGHTS["signal_strength"] * r["signal_strength_score"]
                + RULE_TRIAGE_WEIGHTS["source_weight"] * source_weight
            ),
            2,
        )

    df = rows_to_polars(rows)
    rule_relevant = sum(1 for r in rows if r.get("is_relevant_heuristic"))
    print(f"Records prepared for classification before final filtering: {len(df):,}")
    print(f"Rule/keyword relevant records: {rule_relevant:,}")
    display(
        df.sort("triage_score", descending=True)
        .select(
            [
                "source",
                "_source_family",
                "event_date",
                "primary_category_rule",
                "direction_rule",
                "direction_risk_score",
                "direction_opportunity_score",
                "is_relevant_heuristic",
                "triage_score",
                "title",
                "url",
            ]
        )
        .head(DEFAULT_DISPLAY_ROWS)
    )

In [ ]:
if ENABLE_AHO_RECORD_AUDIT and not df.is_empty():
    try:
        aho_started = time.time()
        progress_log(
            "aho:start",
            0,
            len(df),
            input_records=f"{len(df):,}",
            include_full_text=INCLUDE_FULL_TEXT_IN_AUDIT,
            audit_text_max_chars=AUDIT_TEXT_MAX_CHARS,
        )

        def normalize_pattern_list(
            patterns: collections.abc.Iterable[Any], limit: int | None = None
        ) -> list[str]:
            """Deduplicate and trim fixed string patterns for contains_any()."""
            out: list[str] = []
            seen: set[str] = set()
            for value in patterns:
                text = str(value or "").strip().lower()
                text = re.sub(r"\s+", " ", text)
                if len(text) < AHO_MIN_PATTERN_CHARS:
                    continue
                if text in seen:
                    continue
                seen.add(text)
                out.append(text)
                if limit is not None and len(out) >= limit:
                    break
            return out

        def build_category_patterns() -> dict[str, list[str]]:
            patterns: dict[str, list[str]] = {}
            for category, terms in SIGNAL_TAXONOMY.items():
                readable_name = str(category).replace("_", " ")
                raw_patterns = [readable_name, *[str(t) for t in terms]]
                patterns[category] = normalize_pattern_list(
                    raw_patterns, AHO_MAX_PATTERNS_PER_GROUP
                )
            patterns.setdefault(
                GENERAL_CATEGORY,
                normalize_pattern_list(
                    [INDUSTRY, *INDUSTRY_TERMS, *ADJACENT_TERMS],
                    AHO_MAX_PATTERNS_PER_GROUP,
                ),
            )
            return patterns

        CATEGORY_PATTERNS = build_category_patterns()
        RELEVANCE_PATTERNS = normalize_pattern_list(
            [INDUSTRY, *INDUSTRY_TERMS, *ADJACENT_TERMS], AHO_MAX_PATTERNS_PER_GROUP
        )
        DIRECTION_PATTERNS_CLEAN = {
            direction: normalize_pattern_list(patterns, AHO_MAX_PATTERNS_PER_GROUP)
            for direction, patterns in DIRECTION_PATTERNS.items()
        }

        def add_contains_any_column(
            frame: pl.DataFrame, out_col: str, patterns: list[str]
        ) -> pl.DataFrame:
            """Use contains_any, with regex fallback for older Polars."""
            if not patterns:
                return frame.with_columns(pl.lit(False).alias(out_col))
            try:
                return frame.with_columns(
                    pl.col("audit_text")
                    .str.contains_any(patterns, ascii_case_insensitive=True)
                    .fill_null(False)
                    .alias(out_col)
                )
            except Exception:
                regex = "(?i)" + "|".join(re.escape(p) for p in patterns)
                return frame.with_columns(
                    pl.col("audit_text")
                    .str.contains(regex)
                    .fill_null(False)
                    .alias(out_col)
                )

        text_cols = [c for c in TEXT_COLUMNS_FOR_AUDIT if c in df.columns]
        if not text_cols:
            text_cols = ["title"] if "title" in df.columns else []
        text_exprs = [pl.col(c).cast(pl.Utf8).fill_null("") for c in text_cols]
        if INCLUDE_FULL_TEXT_IN_AUDIT and "full_text" in df.columns:
            text_exprs.append(
                pl.col("full_text")
                .cast(pl.Utf8)
                .fill_null("")
                .str.slice(0, AUDIT_TEXT_MAX_CHARS)
            )
        if not text_exprs:
            df = df.with_columns(pl.lit("").alias("audit_text"))
        else:
            df = df.with_columns(
                pl.concat_str(text_exprs, separator=" ")
                .str.slice(0, AUDIT_TEXT_MAX_CHARS)
                .str.to_lowercase()
                .alias("audit_text")
            )

        original_audit_rows = len(df)
        short_text_count = int(
            df.select(
                (pl.col("audit_text").str.len_chars() < AHO_MIN_TEXT_CHARS)
                .cast(pl.Int64)
                .sum()
            ).item()
        )
        progress_log(
            "aho:prepare",
            original_audit_rows - short_text_count,
            original_audit_rows,
            eligible_records=f"{original_audit_rows - short_text_count:,}",
            skipped_short_text=f"{short_text_count:,}",
            category_groups=len(CATEGORY_PATTERNS),
            direction_groups=len(DIRECTION_PATTERNS_CLEAN),
        )

        df = add_contains_any_column(df, "aho_relevance_match", RELEVANCE_PATTERNS)

        category_cols: list[str] = []
        for category, patterns in CATEGORY_PATTERNS.items():
            col_name = f"aho_match_{slugify(category, 60)}"
            category_cols.append(col_name)
            df = add_contains_any_column(df, col_name, patterns)

        direction_cols: dict[str, str] = {}
        for direction, patterns in DIRECTION_PATTERNS_CLEAN.items():
            col_name = f"aho_direction_{slugify(direction, 40)}"
            direction_cols[direction] = col_name
            df = add_contains_any_column(df, col_name, patterns)

        if category_cols:
            df = df.with_columns(
                pl.sum_horizontal(
                    [pl.col(c).cast(pl.Int64) for c in category_cols]
                ).alias("aho_category_match_count")
            )
        else:
            df = df.with_columns(pl.lit(0).alias("aho_category_match_count"))

        if direction_cols:
            df = df.with_columns(
                pl.sum_horizontal(
                    [pl.col(c).cast(pl.Int64) for c in direction_cols.values()]
                ).alias("aho_direction_match_count")
            )
        else:
            df = df.with_columns(pl.lit(0).alias("aho_direction_match_count"))

        category_col_by_name = {
            cat: f"aho_match_{slugify(cat, 60)}" for cat in CATEGORY_PATTERNS
        }
        result_rows = []
        for r in df.to_dicts():
            matched_categories = [
                cat
                for cat in CATEGORY_PRIORITY
                if bool(r.get(category_col_by_name.get(cat, "")))
            ]
            matched_categories += [
                cat
                for cat in CATEGORY_PATTERNS
                if cat not in matched_categories
                and bool(r.get(category_col_by_name.get(cat, "")))
            ]
            primary = (
                matched_categories[0] if matched_categories else UNCATEGORIZED_CATEGORY
            )

            direction_info = direction_evidence(r.get("audit_text", ""))
            direction = direction_info["direction"]
            neutral_match = bool(r.get(direction_cols.get("neutral", "")))
            if direction == "neutral" and neutral_match:
                direction = "neutral"

            relevance_score = 1.0 if bool(r.get("aho_relevance_match")) else 0.0
            category_score = min(
                1.0,
                to_float(r.get("aho_category_match_count"), 0.0)
                / max(AHO_CATEGORY_SCORE_NORMALIZER, 1e-9),
            )
            direction_score = min(
                1.0,
                max(
                    to_float(direction_info["risk_score"], 0.0),
                    to_float(direction_info["opportunity_score"], 0.0),
                )
                / max(AHO_DIRECTION_SCORE_NORMALIZER, 1e-9),
            )
            confidence = float(
                np.clip(
                    AHO_CONFIDENCE_WEIGHTS["relevance"] * relevance_score
                    + AHO_CONFIDENCE_WEIGHTS["category"] * category_score
                    + AHO_CONFIDENCE_WEIGHTS["direction"] * direction_score,
                    0,
                    1,
                )
            )
            total_match_groups = (
                int(to_float(r.get("aho_category_match_count"), 0.0))
                + int(bool(r.get("aho_relevance_match")))
                + int(to_float(r.get("aho_direction_match_count"), 0.0))
            )
            is_relevant = bool(
                total_match_groups >= AHO_MIN_TOTAL_MATCH_GROUPS
                and (relevance_score > 0 or category_score > 0)
            )
            result_rows.append(
                {
                    "record_id": r["record_id"],
                    "aho_is_relevant": is_relevant,
                    "aho_relevance_score": relevance_score,
                    "aho_primary_category": primary,
                    "aho_category_score": category_score,
                    "aho_direction": direction,
                    "aho_direction_score": direction_score,
                    "aho_direction_risk_score": direction_info["risk_score"],
                    "aho_direction_opportunity_score": direction_info[
                        "opportunity_score"
                    ],
                    "aho_confidence": confidence,
                    "aho_total_match_groups": total_match_groups,
                    "aho_matched_categories": matched_categories,
                }
            )

        aho_audit = rows_to_polars(result_rows, columns=["record_id"])
        df = df.drop(
            [
                c
                for c in ["audit_text", *category_cols, *direction_cols.values()]
                if c in df.columns
            ]
        )
        df = df.join(aho_audit, on="record_id", how="left")

        print(
            "Polars/Aho-Corasick audit completed for "
            f"{len(aho_audit):,} records without scikit-learn or LLMs."
        )
        if "aho_is_relevant" in aho_audit.columns:
            relevant_count = int(
                aho_audit.select(pl.col("aho_is_relevant").cast(pl.Int64).sum()).item()
            )
            print(f"Aho relevant records: {relevant_count:,}")
        print(f"Aho elapsed: {format_duration(time.time() - aho_started)}")
        if not aho_audit.is_empty():
            display(
                aho_audit.sort("aho_confidence", descending=True).head(AHO_DISPLAY_ROWS)
            )

    except Exception as exc:
        print(f"Skipping Polars/Aho-Corasick audit: {type(exc).__name__}: {exc}")
else:
    print("Aho record audit disabled. Set ENABLE_AHO_RECORD_AUDIT=true to enable it.")


df = ensure_columns(
    df,
    [
        "aho_is_relevant",
        "aho_relevance_score",
        "aho_primary_category",
        "aho_category_score",
        "aho_direction",
        "aho_direction_score",
        "aho_direction_risk_score",
        "aho_direction_opportunity_score",
        "aho_confidence",
        "aho_total_match_groups",
        "aho_matched_categories",
    ],
)

final_rows = []
for r in df.to_dicts():
    heuristic = bool(r.get("is_relevant_heuristic"))
    aho_rel = bool(r.get("aho_is_relevant"))
    r["is_relevant_final"] = heuristic or aho_rel

    aho_cat_score = to_float(r.get("aho_category_score"), 0.0)
    if (not is_missing(r.get("aho_primary_category"))) and (
        (not heuristic)
        or (r.get("primary_category") in [None, UNCATEGORIZED_CATEGORY])
        or (aho_cat_score > 0)
    ):
        r["primary_category"] = r.get("aho_primary_category")

    combined_risk_score = max(
        to_float(r.get("direction_risk_score"), 0.0),
        to_float(r.get("aho_direction_risk_score"), 0.0),
    )
    combined_opportunity_score = max(
        to_float(r.get("direction_opportunity_score"), 0.0),
        to_float(r.get("aho_direction_opportunity_score"), 0.0),
    )
    combined_margin = abs(combined_risk_score - combined_opportunity_score)
    if (
        max(combined_risk_score, combined_opportunity_score)
        < DIRECTION_MIN_EVIDENCE_SCORE
        or combined_margin < DIRECTION_RESOLUTION_MARGIN
    ):
        r["direction"] = "neutral"
    elif combined_risk_score > combined_opportunity_score:
        r["direction"] = "risk"
    else:
        r["direction"] = "opportunity"
    r["direction_evidence_margin"] = round(combined_margin, 3)

    r["aho_triage_component"] = round(
        TRIAGE_SCORE_SCALE
        * (
            AHO_TRIAGE_WEIGHTS["aho_confidence"]
            * to_float(r.get("aho_confidence"), 0.0)
            + AHO_TRIAGE_WEIGHTS["freshness"] * to_float(r.get("freshness_score"), 0.0)
            + AHO_TRIAGE_WEIGHTS["source_weight"]
            * to_float(r.get("_source_weight"), DEFAULT_SOURCE_WEIGHT)
        ),
        2,
    )
    r["triage_score"] = round(
        max(
            to_float(r.get("triage_score"), 0.0),
            to_float(r.get("aho_triage_component"), 0.0),
        ),
        2,
    )

    if heuristic and aho_rel:
        r["classification_method"] = CLASSIFICATION_METHOD_LABELS["rules_and_aho"]
    elif heuristic:
        r["classification_method"] = CLASSIFICATION_METHOD_LABELS["rules_only"]
    elif aho_rel:
        r["classification_method"] = CLASSIFICATION_METHOD_LABELS["aho_only"]
    else:
        r["classification_method"] = CLASSIFICATION_METHOD_LABELS["not_relevant"]

    final_rows.append(r)

before_filter = len(final_rows)
relevant_marked = sum(1 for r in final_rows if r.get("is_relevant_final"))
progress_log(
    "aho:final_filter",
    0,
    before_filter,
    candidate_records=f"{before_filter:,}",
    relevant_marked=f"{relevant_marked:,}",
)
final_rows = [r for r in final_rows if r.get("is_relevant_final")]
df = rows_to_polars(final_rows)
progress_log(
    "aho:final_filter",
    before_filter,
    before_filter,
    kept_records=f"{len(df):,}",
    dropped_records=f"{before_filter - len(df):,}",
)
print(
    f"Relevant records after final rules + Aho filter: {len(df):,} of {before_filter:,}"
)

if not df.is_empty():
    display(
        df.group_by("classification_method")
        .len(name="records")
        .sort("records", descending=True)
    )
    display(
        df.sort("triage_score", descending=True)
        .select(
            [
                "source",
                "_source_family",
                "event_date",
                "primary_category",
                "direction",
                "classification_method",
                "aho_relevance_score",
                "aho_category_score",
                "triage_score",
                "title",
                "url",
            ]
        )
        .head(FINAL_CLASSIFICATION_DISPLAY_ROWS)
    )
else:
    print("No relevant records after final filtering.")

In [ ]:
if clean_df.is_empty():
    denominator = pl_empty()
else:
    denominator = (
        clean_df.group_by(
            ["_run_date", "_industry", "_source_family", "source", "_query_lens"]
        )
        .agg(
            [
                pl.col("record_fingerprint").n_unique().alias("records_deduped"),
                pl.col("domain").n_unique().alias("unique_domains"),
            ]
        )
        .rename(
            {
                "_run_date": "run_date",
                "_industry": "industry",
                "_source_family": "source_family",
                "_query_lens": "query_lens",
            }
        )
    )

if df.is_empty():
    numerator = pl.DataFrame(
        {
            "run_date": [],
            "industry": [],
            "source_family": [],
            "source": [],
            "query_lens": [],
            "primary_category": [],
            "direction": [],
            "signal_records": [],
            "avg_triage_score": [],
            "median_triage_score": [],
            "max_triage_score": [],
            "example_title": [],
            "example_url": [],
        }
    )
else:
    numerator = (
        df.group_by(
            [
                "_run_date",
                "_industry",
                "_source_family",
                "source",
                "_query_lens",
                "primary_category",
                "direction",
            ]
        )
        .agg(
            [
                pl.col("record_fingerprint").n_unique().alias("signal_records"),
                pl.col("triage_score").mean().alias("avg_triage_score"),
                pl.col("triage_score").median().alias("median_triage_score"),
                pl.col("triage_score").max().alias("max_triage_score"),
                pl.col("title").drop_nulls().first().alias("example_title"),
                pl.col("url").drop_nulls().first().alias("example_url"),
            ]
        )
        .rename(
            {
                "_run_date": "run_date",
                "_industry": "industry",
                "_source_family": "source_family",
                "_query_lens": "query_lens",
            }
        )
    )

panel = (
    numerator.join(
        denominator,
        on=["run_date", "industry", "source_family", "source", "query_lens"],
        how="left",
    )
    if not numerator.is_empty()
    else numerator
)
panel = (
    panel.with_columns(
        [
            (
                pl.col("signal_records")
                / pl.when(pl.col("records_deduped") == 0)
                .then(None)
                .otherwise(pl.col("records_deduped"))
            ).alias("signal_rate")
            if "records_deduped" in panel.columns
            else pl.lit(None).alias("signal_rate"),
            pl.lit(RUN_ID).alias("run_id"),
            pl.lit(START_DATE).alias("window_start_date"),
            pl.lit(END_DATE).alias("window_end_date"),
            pl.lit(RUN_MODE).alias("run_mode"),
        ]
    )
    if not panel.is_empty() or panel.columns
    else panel
)

coverage_summary = (
    coverage_df.group_by(["source_family", "source", "lens"])
    .agg(
        [
            pl.col("spec_idx").count().alias("specs_attempted"),
            pl.col("status")
            .is_in([FETCH_CACHE_STATUS_MISS, FETCH_CACHE_STATUS_HIT])
            .cast(pl.Int64)
            .sum()
            .alias("specs_success"),
            pl.col("raw_rows").sum().alias("raw_rows"),
            (pl.col("status") == FETCH_CACHE_STATUS_FAILED)
            .cast(pl.Int64)
            .sum()
            .alias("failed_specs"),
        ]
    )
    .rename({"lens": "query_lens"})
)
if not panel.is_empty():
    panel = panel.join(
        coverage_summary, on=["source_family", "source", "query_lens"], how="left"
    )

panel.write_csv(PANEL_CSV)
print(f"Panel rows this run: {len(panel):,}; wrote {PANEL_CSV}")
if not panel.is_empty():
    display(panel.sort("signal_rate", descending=True).head(20))
else:
    display(panel)

In [ ]:
def safe_read_csv(path: Path) -> pl.DataFrame:
    return safe_read_csv_polars(path)


history_old = safe_read_csv(PANEL_HISTORY_CSV)
if not history_old.is_empty() and "run_id" in history_old.columns:
    history_old = history_old.filter(pl.col("run_id").cast(pl.Utf8) != RUN_ID)

history = concat_polars([history_old, panel])
if not history.is_empty() and "run_date" in history.columns:
    history = history.with_columns(
        pl.col("run_date").cast(pl.Utf8).str.slice(0, 10).alias("run_date")
    )
history.write_csv(PANEL_HISTORY_CSV)
print(f"Panel history rows: {len(history):,}; wrote {PANEL_HISTORY_CSV}")

current_keys = [
    "industry",
    "source_family",
    "source",
    "query_lens",
    "primary_category",
    "direction",
]
current_date_obj = parse_date(RUN_DATE)
lookback_start = current_date_obj - timedelta(days=BASELINE_LOOKBACK_DAYS)

if history.is_empty() or "run_id" not in history.columns:
    hist_prior = pl_empty()
else:
    hist_prior = (
        history.filter(pl.col("run_id").cast(pl.Utf8) != RUN_ID)
        .with_columns(
            pl.col("run_date")
            .cast(pl.Utf8)
            .str.slice(0, 10)
            .str.strptime(pl.Date, format="%Y-%m-%d", strict=False)
            .alias("run_date_dt")
        )
        .filter(
            (pl.col("run_date_dt") >= lookback_start)
            & (pl.col("run_date_dt") < current_date_obj)
        )
    )

if hist_prior.is_empty():
    panel_stats = (
        panel.with_columns(
            [
                pl.lit(None).alias("baseline_signal_records"),
                pl.lit(None).alias("baseline_records_deduped"),
                pl.lit(None).alias("baseline_days"),
                pl.lit(None).alias("baseline_avg_rate"),
                pl.lit(None).alias("baseline_rate"),
            ]
        )
        if not panel.is_empty()
        else panel
    )
else:
    baseline = (
        hist_prior.group_by(current_keys)
        .agg(
            [
                pl.col("signal_records")
                .cast(pl.Float64, strict=False)
                .sum()
                .alias("baseline_signal_records"),
                pl.col("records_deduped")
                .cast(pl.Float64, strict=False)
                .sum()
                .alias("baseline_records_deduped"),
                pl.col("run_date").n_unique().alias("baseline_days"),
                pl.col("signal_rate")
                .cast(pl.Float64, strict=False)
                .mean()
                .alias("baseline_avg_rate"),
            ]
        )
        .with_columns(
            (
                pl.col("baseline_signal_records")
                / pl.when(pl.col("baseline_records_deduped") == 0)
                .then(None)
                .otherwise(pl.col("baseline_records_deduped"))
            ).alias("baseline_rate")
        )
    )
    panel_stats = panel.join(baseline, on=current_keys, how="left")

prior_days = (
    0
    if hist_prior.is_empty()
    else int(hist_prior.select(pl.col("run_date").n_unique()).item())
)
print("Prior baseline days available:", prior_days)
display(panel_stats.head())

In [ ]:
def normal_survival_two_sided(z: float) -> float:
    if is_missing(z):
        return np.nan
    return math.erfc(abs(float(z)) / math.sqrt(2))


def two_proportion_z_test(
    x1: float, n1: float, x0: float, n0: float
) -> tuple[float, float]:
    if (
        any(is_missing(v) for v in [x1, n1, x0, n0])
        or to_float(n1) <= 0
        or to_float(n0) <= 0
    ):
        return np.nan, np.nan
    x1, n1, x0, n0 = map(float, [x1, n1, x0, n0])
    p1, p0 = x1 / n1, x0 / n0
    pooled = (x1 + x0) / (n1 + n0)
    se = math.sqrt(max(pooled * (1 - pooled) * (1 / n1 + 1 / n0), 0))
    if se == 0:
        return np.nan, np.nan
    z = (p1 - p0) / se
    return z, normal_survival_two_sided(z)


def wilson_ci(x: float, n: float, z: float = WILSON_Z) -> tuple[float, float]:
    if is_missing(x) or is_missing(n) or to_float(n) <= 0:
        return np.nan, np.nan
    x, n = float(x), float(n)
    p = x / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - half), min(1, center + half)


def benjamini_hochberg_list(pvalues: list[Any]) -> list[float]:
    indexed = [
        (i, to_float(p, np.nan))
        for i, p in enumerate(pvalues)
        if not is_missing(p) and math.isfinite(to_float(p, np.nan))
    ]
    q_values = [np.nan] * len(pvalues)
    m = len(indexed)
    if m == 0:
        return q_values
    indexed.sort(key=lambda x: x[1])
    raw = [
        (idx, min(max(p * m / rank, 0), 1))
        for rank, (idx, p) in enumerate(indexed, start=1)
    ]
    min_so_far = 1.0
    for idx, q in reversed(raw):
        min_so_far = min(min_so_far, q)
        q_values[idx] = min_so_far
    return q_values


stats_rows = panel_stats.to_dicts()
p_values = []
for r in stats_rows:
    z, p = two_proportion_z_test(
        r.get("signal_records"),
        r.get("records_deduped"),
        r.get("baseline_signal_records"),
        r.get("baseline_records_deduped"),
    )
    r["z_score"] = z
    r["p_value"] = p
    p_values.append(p)

q_values = benjamini_hochberg_list(p_values)
for r, q in zip(stats_rows, q_values, strict=True):
    ci_low, ci_high = wilson_ci(r.get("signal_records"), r.get("records_deduped"))
    r["q_value_fdr"] = q
    r["signal_rate_ci_low"] = ci_low
    r["signal_rate_ci_high"] = ci_high
    r["rate_lift_vs_baseline"] = safe_ratio(
        r.get("signal_rate"), r.get("baseline_rate")
    )
    r["enough_current_records"] = (
        to_float(r.get("records_deduped"), 0) >= MIN_RECORDS_FOR_STATS
    )
    r["enough_baseline"] = (
        to_float(r.get("baseline_days"), 0) >= MIN_BASELINE_DAYS
    ) and (to_float(r.get("baseline_records_deduped"), 0) >= MIN_RECORDS_FOR_STATS)
    r["statistically_unusual"] = bool(
        r["enough_current_records"]
        and r["enough_baseline"]
        and (not is_missing(q))
        and q <= FDR_ALPHA
        and to_float(r.get("signal_rate"), -1)
        > to_float(r.get("baseline_rate"), np.inf)
    )

stats = rows_to_polars(stats_rows)
stat_count = (
    int(stats.select(pl.col("statistically_unusual").cast(pl.Int64).sum()).item())
    if "statistically_unusual" in stats.columns and not stats.is_empty()
    else 0
)
print(f"Statistically unusual candidates: {stat_count}")
if not stats.is_empty():
    display(
        stats.sort(
            ["statistically_unusual", "q_value_fdr", "rate_lift_vs_baseline"],
            descending=[True, False, True],
        ).head(STAT_CANDIDATE_DISPLAY_ROWS)
    )
else:
    display(stats)

In [ ]:
if df.is_empty():
    corroboration = pl.DataFrame(
        {
            "primary_category": [],
            "direction": [],
            "corroborating_records": [],
            "source_family_count": [],
            "source_count": [],
            "domains": [],
            "avg_triage_score": [],
        }
    )
else:
    corroboration = df.group_by(["primary_category", "direction"]).agg(
        [
            pl.col("record_fingerprint").n_unique().alias("corroborating_records"),
            pl.col("_source_family").n_unique().alias("source_family_count"),
            pl.col("source").n_unique().alias("source_count"),
            pl.col("domain").n_unique().alias("domains"),
            pl.col("triage_score").mean().alias("avg_triage_score"),
        ]
    )

stats = (
    stats.join(corroboration, on=["primary_category", "direction"], how="left")
    if not stats.is_empty()
    else stats
)
stats_rows = stats.to_dicts()


def claim_level_dict(row: dict[str, Any]) -> str:
    is_corroborated = (
        to_float(row.get("source_family_count"), 0) >= MIN_CORROBORATING_SOURCE_FAMILIES
    )
    row["is_corroborated"] = bool(is_corroborated)
    if bool(row.get("statistically_unusual")) and is_corroborated:
        return CLAIM_LEVELS["stat_and_corroborated"]
    if bool(row.get("statistically_unusual")):
        return CLAIM_LEVELS["stat_single_family"]
    if (
        is_corroborated
        and to_float(row.get("corroborating_records"), 0) >= MIN_CORROBORATING_RECORDS
    ):
        return CLAIM_LEVELS["corroborated_directional"]
    if to_float(row.get("records_deduped"), 0) >= MIN_RECORDS_FOR_STATS:
        return CLAIM_LEVELS["directional_watchlist"]
    return CLAIM_LEVELS["low_volume"]


for r in stats_rows:
    r["claim_level"] = claim_level_dict(r)

stats = rows_to_polars(stats_rows)
stats.write_csv(SIGNIFICANT_SIGNALS_CSV)

print(f"Wrote {SIGNIFICANT_SIGNALS_CSV}")
if not stats.is_empty():
    display(
        stats.sort(
            ["claim_level", "q_value_fdr", "signal_rate"],
            descending=[False, False, True],
        ).head(CORROBORATION_DISPLAY_ROWS)
    )
else:
    display(stats)

In [ ]:
if df.is_empty():
    review_sample = pl.DataFrame({col: [] for col in REVIEW_COLUMNS})
else:
    top_review = df.sort("triage_score", descending=True).head(REVIEW_TOP_N)
    random_n = min(REVIEW_RANDOM_N, len(df))
    random_review = (
        df.sample(n=random_n, seed=REVIEW_RANDOM_SEED) if random_n else pl_empty()
    )
    review_sample = concat_polars([top_review, random_review]).unique(
        subset=["record_id"], keep="first"
    )

review_sample = ensure_columns(review_sample, REVIEW_COLUMNS, None)
review_export = dataframe_for_csv(
    review_sample.select(REVIEW_COLUMNS), list_separator="; "
)
review_export = ensure_columns(review_export, REVIEW_COLUMNS + HUMAN_LABEL_COLUMNS, "")

existing_review = safe_read_csv(HUMAN_REVIEW_CSV)
if not existing_review.is_empty() and "record_id" in existing_review.columns:
    review_export = concat_polars([existing_review, review_export]).unique(
        subset=["record_id"], keep="first"
    )
review_export.write_csv(HUMAN_REVIEW_CSV)
print(f"Wrote/updated human review queue: {HUMAN_REVIEW_CSV}")
display(review_export.tail(CLEAN_RECORDS_DISPLAY_ROWS))

if not review_export.is_empty() and "human_relevant" in review_export.columns:
    labeled = review_export.filter(
        pl.col("human_relevant")
        .cast(pl.Utf8)
        .str.to_lowercase()
        .is_in(list(LABEL_VALID_VALUES))
    )
else:
    labeled = pl_empty()

if not labeled.is_empty():
    rel = labeled.select(
        pl.col("human_relevant")
        .cast(pl.Utf8)
        .str.to_lowercase()
        .is_in(list(LABEL_TRUE_VALUES))
        .cast(pl.Float64)
        .mean()
    ).item()
    print(f"Labeled examples: {len(labeled)}; estimated relevance precision: {rel:.2%}")
else:
    print(
        "No human labels yet. Fill human_relevant/human_true_signal "
        "columns to measure precision."
    )

In [ ]:
if not EXTERNAL_OUTCOMES_CSV.exists():
    rows_to_polars(EXTERNAL_OUTCOME_TEMPLATE_ROWS).write_csv(EXTERNAL_OUTCOMES_CSV)
    print(f"Created external outcome template: {EXTERNAL_OUTCOMES_CSV}")
else:
    print(f"External outcome template exists: {EXTERNAL_OUTCOMES_CSV}")

outcomes = safe_read_csv(EXTERNAL_OUTCOMES_CSV)
valid_outcomes = (
    outcomes.filter(
        pl.col("outcome_date").cast(pl.Utf8).str.contains(DATE_YYYY_MM_DD_REGEX)
    )
    if not outcomes.is_empty() and "outcome_date" in outcomes.columns
    else pl_empty()
)
print(f"Validated outcome rows available: {len(valid_outcomes)}")

In [ ]:
watchlist = (
    df.sort(["triage_score", "_source_weight"], descending=[True, True])
    if not df.is_empty()
    else pl_empty()
)
watchlist = ensure_columns(watchlist, WATCHLIST_EXPORT_COLUMNS, None)
watchlist_export = dataframe_for_csv(
    watchlist.select(WATCHLIST_EXPORT_COLUMNS), list_separator="; "
)
clean_records_export = dataframe_for_csv(df, list_separator="; ")

watchlist_export.write_csv(WATCHLIST_CSV)
clean_records_export.write_csv(CLEAN_RECORDS_CSV)

print(f"Wrote clean records: {CLEAN_RECORDS_CSV}")
print(f"Wrote analyst watchlist: {WATCHLIST_CSV}")

display(watchlist_export.head(WATCHLIST_DISPLAY_ROWS))

In [ ]:
def md_link(title: str, url: str) -> str:
    title = compact_spaces(str(title)) or "Untitled"
    if isinstance(url, str) and url.startswith("http"):
        return f"[{title}]({url})"
    return title


def records_section(name: str, data: pl.DataFrame, n: int = 10) -> str:
    lines = [f"## {name}", ""]
    if data.is_empty():
        return "\n".join([*lines, "No records found.", ""])
    for r in data.head(n).to_dicts():
        score = to_float(r.get("triage_score"), 0)
        source = r.get("source")
        source_family = r.get("_source_family")
        lines.append(
            f"- **{score:.2f}** | {source} / {source_family} | "
            f"{r.get('event_date')} | {r.get('primary_category')} | "
            f"{r.get('direction')} — {md_link(r.get('title'), r.get('url'))}"
        )
    lines.append("")
    return "\n".join(lines)


def stats_section(data: pl.DataFrame, n: int = 15) -> str:
    lines = ["## Statistical signal candidates", ""]
    if data.is_empty():
        return "\n".join([*lines, "No panel rows available.", ""])
    candidate = data.sort(
        ["statistically_unusual", "is_corroborated", "q_value_fdr", "signal_rate"],
        descending=[True, True, False, True],
    ).head(n)
    for r in candidate.to_dicts():
        qv = r.get("q_value_fdr")
        qv_txt = "NA" if is_missing(qv) else f"{float(qv):.4f}"
        base = r.get("baseline_rate")
        base_txt = "NA" if is_missing(base) else f"{float(base):.2%}"
        signal_records = int(to_float(r.get("signal_records"), 0))
        records_deduped = int(to_float(r.get("records_deduped"), 0))
        source_family_count = int(to_float(r.get("source_family_count"), 0))
        lines.append(
            f"- **{r.get('claim_level')}** | "
            f"{r.get('source_family')}/{r.get('source')} | "
            f"{r.get('primary_category')} / {r.get('direction')} | "
            f"rate {to_float(r.get('signal_rate'), 0):.2%} "
            f"vs baseline {base_txt}; q={qv_txt}; "
            f"families={source_family_count}; "
            f"records={signal_records}/{records_deduped}. "
            f"Example: {md_link(r.get('example_title', ''), r.get('example_url', ''))}"
        )
    lines.append("")
    return "\n".join(lines)


source_summary = (
    coverage_df.group_by(["source_family", "source", "status"])
    .agg(
        [
            pl.col("spec_idx").count().alias("specs"),
            pl.col("raw_rows").sum().alias("raw_rows"),
        ]
    )
    .sort(["source_family", "source", "status"])
)
source_lines = ["## Source coverage", ""]
for r in source_summary.to_dicts():
    specs = int(to_float(r["specs"], 0))
    raw_rows = int(to_float(r["raw_rows"], 0))
    source_lines.append(
        f"- **{r['source_family']}/{r['source']}** "
        f"{r['status']}: {specs} specs, {raw_rows} raw rows"
    )
source_lines.append("")

prior_days_for_report = (
    0
    if hist_prior.is_empty()
    else int(hist_prior.select(pl.col("run_date").n_unique()).item())
)
report = [
    f"# Industry Signal Report: {INDUSTRY}",
    "",
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"Run id: `{RUN_ID}`",
    (
        f"Mode: `{RUN_MODE}`; backfill: `{BACKFILL_MODE}`; "
        f"window: {START_DATE} to {END_DATE}"
    ),
    "",
    "## Reality check",
    "",
    REPORT_REALITY_CHECK_TEXT,
    "",
    (
        f"Raw records: {len(raw_df):,}; deduped records: {len(clean_df):,}; "
        f"relevant records: {len(df):,}"
    ),
    (
        f"Panel history rows: {len(history):,}; prior baseline days "
        f"available: {prior_days_for_report}"
    ),
    "",
    "\n".join(source_lines),
    stats_section(stats, REPORT_TOP_N_STATS),
    records_section("Top analyst watchlist", watchlist, REPORT_TOP_N_WATCHLIST),
]

for section in REPORT_CATEGORY_SECTIONS:
    filtered = (
        watchlist.filter(pl.col(section["column"]) == section["value"])
        if not watchlist.is_empty() and section["column"] in watchlist.columns
        else pl_empty()
    )
    report.append(records_section(section["name"], filtered, section["n"]))

report.extend(
    [
        "## Files written",
        "",
        f"- `{COMBINED_RAW_JSONL}`",
        f"- `{COVERAGE_CSV}`",
        f"- `{CLEAN_RECORDS_CSV}`",
        f"- `{PANEL_CSV}`",
        f"- `{PANEL_HISTORY_CSV}`",
        f"- `{SIGNIFICANT_SIGNALS_CSV}`",
        f"- `{WATCHLIST_CSV}`",
        f"- `{HUMAN_REVIEW_CSV}`",
        f"- `{EXTERNAL_OUTCOMES_CSV}`",
    ]
)

REPORT_MD.write_text("\n".join(report), encoding=JSON_ENCODING)
print(f"Wrote report: {REPORT_MD}")
print("\n".join(report[:40]))

In [ ]:
import matplotlib.pyplot as plt

if df.is_empty():
    print("No relevant records to visualize.")
else:
    source_counts = df.group_by("source").len(name="records").sort("records")
    plt.figure(figsize=PLOT_FIGSIZE_SOURCE)
    plt.barh(source_counts["source"].to_list(), source_counts["records"].to_list())
    plt.title("Relevant records by source")
    plt.xlabel("records")
    plt.ylabel("source")
    plt.tight_layout()
    plt.show()

    category_counts = (
        df.group_by("primary_category").len(name="records").sort("records")
    )
    plt.figure(figsize=PLOT_FIGSIZE_CATEGORY)
    plt.barh(
        category_counts["primary_category"].to_list(),
        category_counts["records"].to_list(),
    )
    plt.title("Relevant records by category")
    plt.xlabel("records")
    plt.ylabel("category")
    plt.tight_layout()
    plt.show()

    if not panel.is_empty():
        panel_plot = (
            panel.with_columns(
                (
                    pl.col("primary_category").cast(pl.Utf8)
                    + " / "
                    + pl.col("direction").cast(pl.Utf8)
                ).alias("category_direction")
            )
            .group_by("category_direction")
            .agg(pl.col("signal_rate").mean().alias("avg_signal_rate"))
            .sort("avg_signal_rate")
            .tail(15)
        )
        plt.figure(figsize=PLOT_FIGSIZE_SIGNAL_RATE)
        plt.barh(
            panel_plot["category_direction"].to_list(),
            panel_plot["avg_signal_rate"].to_list(),
        )
        plt.title("Average signal rate by category/direction")
        plt.xlabel("signal rate")
        plt.ylabel("category / direction")
        plt.tight_layout()
        plt.show()